# PKG customer periodicity: regular payments, expectations and forecasts

**PNC Treasury Management · DSI · Payment Knowledge Graph**

Every customer's payments are split into **streams** (customer × direction × counterparty) and each stream is followed
month by month: is it regular, at what cadence, when and how much is the next payment due, did it come late, early,
short or not at all, and how much will it bring next month and next quarter. This replaces the periodicity columns of
`pkg_customer_metrics_v4` (which used a 26-payment median and days ÷ median gap) with the methods that held up in
`pkg_regular_forecast_check`: projected payment dates, the median of the last 3 payments (the mean for daily and
internal streams), the spread of the history next to every amount, and intervals calibrated on the bank's own past
errors.

**Tables** (database `DB`, partitioned by `month`):

| table | one row per | use |
|---|---|---|
| `pkg_cust_reg_stream` | stream × month, streams due or live (regular, not stopped) or resumed | contagion events, bottom-up liquidity forecasts, counterparty and bank (FI) views |
| `pkg_cust_reg_month` | customer × direction × side × cadence × month (`irregular` = everything not expected) | cadence mix, events by cadence |
| `pkg_cust_reg_total` | customer × direction × side × month | liquidity: coverage, expected vs received, next month / quarter forecast with intervals |
| `pkg_cust_reg_calib` | calibration group × month | the predictor, bias correction, interval quantiles and stop rates each month used (from earlier months only) |
| `pkg_cust_reg_dictionary` | column | definitions |

**Side:** `ext` external counterparty, `int` another PNC customer (book to book), `own` the customer's own-name account
at another bank (self-pay match of customer metrics). The customer's own PNC accounts (`is_self`) are left out.

**Regular** (same rules as v4): at least `MIN_PAYMENTS` payment days in the last `REG_WINDOW` months, a known cadence
from the median gap and ≥ `REG_MIN` of gaps on time (|gap − median| ≤ max(`TOL_MIN`, `TOL_FRAC` × median); daily: the
next business day). **Due** in a month = regular and not stopped at the end of the month before. **Stopped** = silent
for `SKIP_MULT` × the median gap (daily: `DAILY_STOP_BD` business days).

**Expectation for a month** (made from the history before it): payment dates projected from the last payment —
monthly, bimonthly and quarterly on the usual day of the month 1, 2 or 3 months on; weekly and biweekly every median
gap; a date on a weekend or Fed holiday moves to the next business day; daily = business days × the stream's share of
business days with a payment over the last 3 months — times the amount per payment × a bias correction. The amount
per payment is one of four predictors: median of the last `AMT_RECENT` payments, mean of the history, median of the
history, last payment. Each calibration group uses the predictor with the lowest past dollar error after its own bias
correction; the four raw expectations are kept (`exp_med3`, `exp_mean`, `exp_med`, `exp_last`) and §3d compares them.

**Calibration** comes from the bank's own past: before building a month, the due streams of the previous
`CALIB_MONTHS` months (a hash sample of customers) give, per group, the predictor, the bias correction (received ÷
expected) and the 5th / 95th percentiles of received ÷ corrected expected. Groups are tried from fine to coarse
(`CALIB_LEVELS`): direction × side × cadence × rail × amount band × spread × edge, then coarser; each stream uses the
first group with at least `CALIB_MIN_N` rows. *Edge* = a projected payment date within `EDGE_DAYS` of a month boundary,
where a day's shift moves the payment into the next or previous month. Stop rates come per side × cadence × rail, and
customer-level intervals per direction × side × number of streams. Nothing a month uses comes from that month or later.

**Forecasts:** next month per stream if it continues (`fc1_amt`) and with stop rates applied over the next
`FORECAST_MONTHS` months (`fc3_amt`); the month and total tables add stream forecasts × the chance the stream continues,
so their `fc1_amt` is the expected value, with its interval calibrated on past customer-level forecasts against what came.

**v1.0 → v1.1:** enough projected dates for weekly streams over 3 months (v1.0 cut off at 12 and under-forecast
`fc3` of weekly streams); bias correction (v1.0 expected about 10% less than came); predictor chosen per group; edge,
rail and amount band in the calibration groups (narrower intervals away from month boundaries); total and month
`fc1_amt` with stop rates; `exp_v4` dropped. A full rebuild (new signature).

**Run order:** after `pkg_customer_metrics_v4.ipynb` (reads its pair table for side, bank and class). Built
sequentially from `HISTORY_START`; a finished month is skipped unless its inputs or settings changed. Each month
needs the stream state of the month before (`KEEP_STATES` kept).

**Sections:** Settings · Setup · §1 inputs and plan · §2 build · §3 QA · §4 dictionary and housekeeping.

## Settings

In [ ]:
# ---- Settings ------------------------------------------------------------------------------------
DB = "bdahd01p_dlcdi1_cdi_tm"
SHUFFLE_PARTITIONS = 2000
PER_PARTITIONS     = 4000     # shuffle partitions of the stream step only (no effect on the numbers); raise if executors are lost
PER_VERSION   = "per-v1.1"

# ---- Months
HISTORY_START = "2024-01"     # first month of the edge tables; the state starts empty here
LAST_MONTH    = None          # None = latest month published by both pkg_edges_refresh and the customer-metrics pair table
REBUILD_FROM  = None          # e.g. "2025-06": rebuild from this month on (needs the state of the month before);
                              # set back to None after the rebuild, also before resuming a stopped rebuild
KEEP_STATES   = 2             # stream states kept (latest months)

# ---- Inputs
T_EDGE_D = f"{DB}.pkg_edge_daily"
T_MAP    = f"{DB}.pkg_cpty_entity_map_v2"
T_PAIR   = f"{DB}.pkg_cust_pair_month_v4"      # side (own-name), main bank and name class per customer × counterparty × month
T_CUST   = f"{DB}.pkg_cust_dim"                # §3 QA only (TM flag)
OUT_PREFIX = "pkg_cust_reg"

# ---- Regular streams (same rules as pkg_customer_metrics_v4)
REG_WINDOW, REG_MAX_DAYS = 24, 26   # history: latest REG_MAX_DAYS payment days within REG_WINDOW months
MIN_PAYMENTS, REG_MIN = 4, 0.70
TOL_FRAC, TOL_MIN = 0.15, 1         # on time: |gap − median gap| ≤ max(TOL_MIN, TOL_FRAC × median gap) days
SKIP_MULT     = 1.75                # a gap ≥ SKIP_MULT × median = skipped; silent that long = stopped
DAILY_STOP_BD = 5                   # daily streams: stopped after this many business days without a payment
FIXED_PCT     = 0.01                # amount-low / -high rules: also beyond median × (1 ∓ FIXED_PCT)
MASS_STOP_MIN = 20                  # payer-wide: the payer stopped ≥ this many customers in the month
CADENCES = [("weekly", 6, 8), ("biweekly", 12, 17), ("monthly", 26, 35), ("quarterly", 55, 66), ("quarterly", 84, 98)]
CAD_GROUPS = ["daily", "weekly", "biweekly", "monthly", "quarterly"]
SIDES = ["ext", "int", "own"]

# ---- Expectation, intervals, forecast
AMT_RECENT     = 3              # med3 = median of the last AMT_RECENT payments
METHODS        = ["med3", "mean", "med", "last"]   # amount per payment: median of the last AMT_RECENT, mean of the history,
                                # median of the history, last payment. Each calibration group takes the one with the lowest
                                # dollar error after bias correction (ties: the earlier one in this list)
MEAN_CADENCES  = ["daily"]      # before calibration exists (first CALIB_MIN_MONTHS months) and for streams no group
MEAN_SIDES     = ["int"]        #   covers: mean for these cadences and sides, med3 otherwise, no bias correction
BIAS_RANGE     = (0.5, 2.0)     # bias correction (a group's past received ÷ expected) kept inside this range
SPREAD_BINS    = [0.0, 0.005, 0.02, 0.05, 0.10, 0.25]   # spread = MAD / median of the history; bins for calibration
AMT_BANDS      = [100, 1_000, 10_000, 100_000, 1_000_000]   # median payment of the history (dollars); bands for calibration
EDGE_DAYS      = 3              # edge = a projected payment date within this many days of a month boundary
CALIB_LEVELS   = [["dir", "side", "cadence", "rail", "amt_band", "spread_bin", "edge"],   # tried in this order: each stream
                  ["dir", "side", "cadence", "spread_bin", "edge"],                       # uses the first group with
                  ["side", "cadence", "edge"],                                            # ≥ CALIB_MIN_N rows
                  ["side", "cadence"]]
CALIB_MONTHS   = 12             # months of past errors behind the calibration
CALIB_MIN_MONTHS = 3            # fewer published months before a month: no calibration for it
CALIB_MIN_N    = 200            # rows a calibration group needs to be used
CALIB_PCT      = 20             # % of customers (hash) read for the calibration (enough for percentiles, much cheaper)
INTERVAL       = (0.05, 0.95)   # interval = expected × these percentiles of past received ÷ expected
FORECAST_MONTHS = 3             # fc3 = next FORECAST_MONTHS months
PRUNE_IDLE_DAYS = 175           # streams with < MIN_PAYMENTS payment days silent longer than this leave the state
                                # (even a quarterly stream is stopped after SKIP_MULT × 98 days)
QA_MONTHS      = 12             # §3: months looked back for accuracy

## Setup

In [ ]:
# ---- Setup ---------------------------------------------------------------------------------------
# Compatibility shims: Cloudera SPARK3 PySpark is older than the NumPy / pandas in the conda env.
import numpy as np
import pandas as pd
import warnings
for _alias, _target in {"object0": np.object_, "bool8": np.bool_, "int0": np.intp, "uint0": np.uintp}.items():
    if not hasattr(np, _alias):
        setattr(np, _alias, _target)
if "bool" not in np.__dict__:
    np.bool = np.bool_
if not hasattr(pd.DataFrame, "iteritems"):
    pd.DataFrame.iteritems = pd.DataFrame.items
if not hasattr(pd.Series, "iteritems"):
    pd.Series.iteritems = pd.Series.items
for _cat in (ResourceWarning, DeprecationWarning, FutureWarning):
    warnings.filterwarnings("ignore", category=_cat)

import calendar, hashlib, json, time, datetime as dt, html as _html
from IPython.display import display, HTML
from pyspark.sql import SparkSession, Window, functions as F

spark = (SparkSession.builder.appName("pkg_cust_periodicity")
         .config("spark.yarn.queue", "root.ua.CIB-AMG.spark")
         .config("spark.executor.memory", "8g").config("spark.driver.memory", "4g")
         .config("spark.driver.maxResultSize", "4g").config("spark.executor.memoryOverhead", "3g")
         .config("spark.executor.cores", "2")
         .config("spark.dynamicAllocation.enabled", "true").config("spark.dynamicAllocation.maxExecutors", "512")
         .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
         .config("spark.sql.execution.arrow.pyspark.enabled", "true")
         .config("spark.sql.adaptive.enabled", "true").config("spark.sql.adaptive.skewJoin.enabled", "true")
         .config("spark.ui.showConsoleProgress", "false")
         .enableHiveSupport().getOrCreate())
spark.conf.set("spark.sql.shuffle.partitions", str(SHUFFLE_PARTITIONS))
spark.conf.set("spark.sql.ansi.enabled", "false")      # ratios with a zero denominator are NULL, not an error
spark.sparkContext.setLogLevel("ERROR")
# The cluster's Atlas lineage hook can't reach its Kafka topic and prints the full query plan on every write; netty /
# TransportResponseHandler print harmless errors when executors are released. Their loggers are switched off.
_QUIET = ("org.apache.atlas", "com.hortonworks.spark.atlas", "com.cloudera.spark.lineage", "org.apache.kafka",
          "org.apache.spark.sql.catalyst.util.package", "org.apache.spark.network", "io.netty")
try:
    _L = spark._jvm.org.apache.logging.log4j
    for _n in _QUIET:
        _L.core.config.Configurator.setLevel(_n, _L.Level.OFF)
except Exception:
    try:
        _L1 = spark._jvm.org.apache.log4j
        for _n in _QUIET:
            _L1.LogManager.getLogger(_n).setLevel(_L1.Level.OFF)
    except Exception as _e:
        print("could not silence the Atlas logs:", str(_e)[:120])
pd.set_option("display.max_rows", 300); pd.set_option("display.max_columns", 80)

# ---- one consolidated HTML output per cell
def tbl(pdf, pct=(), dec=()):
    pdf = pdf.copy()
    fmt = {}
    for c in pdf.columns:
        if c in pct:                                   fmt[c] = "{:.1%}"
        elif c in dec:                                 fmt[c] = "{:,.2f}"
        elif pd.api.types.is_bool_dtype(pdf[c]):       continue
        elif pd.api.types.is_integer_dtype(pdf[c]):    fmt[c] = "{:,}"
        elif pd.api.types.is_float_dtype(pdf[c]):      fmt[c] = "{:,.0f}"
    sty = pdf.style.format(fmt, na_rep="—")
    try:
        sty = sty.hide(axis="index")
    except AttributeError:
        sty = sty.hide_index()
    sty = sty.set_table_styles([{"selector": "th, td", "props": "padding: 2px 8px; font-size: 12px;"}])
    return sty.to_html()
def sec(title, body):
    return f"<h4 style='margin:14px 0 4px 0'>{_html.escape(title)}</h4>{body}"
def para(text):
    return f"<p style='margin:4px 0; font-size:13px'>{_html.escape(text)}</p>"
def out(*parts):
    display(HTML("<div>" + "".join(p for p in parts if p) + "</div>"))

# ---- HDFS (Hadoop FileSystem API through the JVM)
HCONF = spark._jsc.hadoopConfiguration()
JPath = spark._jvm.org.apache.hadoop.fs.Path
fs_for = lambda p: JPath(p).getFileSystem(HCONF)
exists = lambda p: fs_for(p).exists(JPath(p))
def delete(p):
    return fs_for(p).delete(JPath(p), True) if exists(p) else False
def size_gb(p):
    return fs_for(p).getContentSummary(JPath(p)).getLength() / 1e9 if exists(p) else 0.0
def ls_dirs(p):
    if not exists(p):
        return []
    return [s.getPath().getName() for s in fs_for(p).listStatus(JPath(p)) if s.isDirectory()]

TIMES = {}
class timed:
    def __init__(self, name): self.name = name
    def __enter__(self): self.t0 = time.time()
    def __exit__(self, *a): TIMES[self.name] = (time.time() - self.t0) / 60

def month_add(m, k):
    t = int(m[:4]) * 12 + int(m[5:7]) - 1 + k
    return f"{t // 12:04d}-{t % 12 + 1:02d}"
def month_range(a, b):
    out_, m = [], a
    while m <= b:
        out_.append(m); m = month_add(m, 1)
    return out_

def us_bank_holidays(y0, y1):
    """US Federal Reserve holidays (ACH / Fedwire closed). A holiday on a Sunday is observed on Monday; one on a
    Saturday is not observed (the Fed stays open on Friday)."""
    def nth(y, m, wd, k):                      # k-th weekday wd (Mon = 0) of the month; k = -1 → last
        if k > 0:
            d = dt.date(y, m, 1)
            d += dt.timedelta(days=(wd - d.weekday()) % 7 + 7 * (k - 1))
        else:
            d = dt.date(y, m, calendar.monthrange(y, m)[1])
            d -= dt.timedelta(days=(d.weekday() - wd) % 7)
        return d
    out_ = []
    for y in range(y0, y1 + 1):
        fixed = [dt.date(y, 1, 1), dt.date(y, 6, 19), dt.date(y, 7, 4), dt.date(y, 11, 11), dt.date(y, 12, 25)]
        moving = [nth(y, 1, 0, 3), nth(y, 2, 0, 3), nth(y, 5, 0, -1), nth(y, 9, 0, 1), nth(y, 10, 0, 2), nth(y, 11, 3, 4)]
        for d in fixed:
            if d.weekday() == 6:
                d += dt.timedelta(days=1)
            if d.weekday() < 5:
                out_.append(d)
        out_ += moving
    return sorted(out_)
HOLIDAYS = us_bank_holidays(2023, 2028)

def bidx(c):
    """Business-day index of a date column: each business day (Monday–Friday, not a Fed holiday) counts 1; weekends
    and holidays share the next business day's index. So Friday → Monday is 1 business day, Friday → Tuesday after a
    Monday holiday is also 1, and a payment made on a weekend or holiday is on time for a daily stream."""
    n = F.datediff(c, F.to_date(F.lit("1970-01-05")))           # 1970-01-05 is a Monday
    wk = F.floor(n / 7)
    hol = sum(F.when(c > F.to_date(F.lit(h.isoformat())), 1).otherwise(0) for h in HOLIDAYS)
    return (wk * 5 + F.least(n - wk * 7, F.lit(5)) - hol).cast("long")

print(f"Spark {spark.version} | pandas {pd.__version__} | numpy {np.__version__}")

# ---- additions for a monthly build: JSON markers on HDFS, table helpers, calendar
def write_json(p, obj):
    s = fs_for(p).create(JPath(p), True)
    s.write(bytearray(json.dumps(obj, default=str).encode())); s.close()
def read_json(p):
    if not exists(p):
        return None
    s = fs_for(p).open(JPath(p))
    try:
        return json.loads(spark._jvm.org.apache.commons.io.IOUtils.toString(s, "UTF-8"))
    finally:
        s.close()
def read_text(p):
    if not exists(p):
        return ""
    s = fs_for(p).open(JPath(p))
    try:
        return spark._jvm.org.apache.commons.io.IOUtils.toString(s, "UTF-8")
    finally:
        s.close()
def ls_files(p):
    if not exists(p):
        return []
    return sorted((s.getPath().getName(), s.getLen()) for s in fs_for(p).listStatus(JPath(p)) if s.isFile())
def table_exists(t):
    db, name = t.split(".")
    return spark.sql(f"SHOW TABLES IN {db} LIKE '{name}'").count() > 0
def table_location(t):
    return [r[1] for r in spark.sql(f"DESCRIBE TABLE EXTENDED {t}").collect() if r[0] == "Location"][0].rstrip("/")
def md5(o):
    return hashlib.md5(json.dumps(o, sort_keys=True, default=str).encode()).hexdigest()[:12]

EPOCH, MONDAY0 = dt.date(1970, 1, 1), dt.date(1970, 1, 5)
def bidx_py(d):
    n = (d - MONDAY0).days
    return n // 7 * 5 + min(n % 7, 5) - sum(1 for h in HOLIDAYS if d > h)
def month_first(m):
    return dt.date(int(m[:4]), int(m[5:7]), 1)
def month_last(m):
    return dt.date(int(m[:4]), int(m[5:7]), calendar.monthrange(int(m[:4]), int(m[5:7]))[1])
def month_idx(m):
    return int(m[:4]) * 12 + int(m[5:7]) - 1
def business_days(m):
    d, n, end = month_first(m), 0, month_last(m)
    while d <= end:
        n += d.weekday() < 5 and d not in HOLIDAYS
        d += dt.timedelta(days=1)
    return n

## §1 Inputs and plan

Checks the inputs, works out the months (published by `pkg_edges_refresh` and present in the pair table) and which of
them are already built with the current settings and inputs. A month is done when its marker carries the current
signature, the fingerprint of its inputs (edge build marker and pair-table files) and the build id of the month before.

In [ ]:
# ---- §1 Inputs and plan ---------------------------------------------------------------------------
DB_LOC = [r[1] for r in spark.sql(f"DESCRIBE DATABASE {DB}").collect() if r[0].lower() == "location"][0].rstrip("/")
TBL = {k: f"{DB}.{OUT_PREFIX}_{k}" for k in ("stream", "month", "total", "calib", "dictionary")}
LOC = {k: f"{DB_LOC}/{OUT_PREFIX}_{k}" for k in TBL}
STATE, WORK = f"{DB_LOC}/_state_{OUT_PREFIX}", f"{DB_LOC}/_work_{OUT_PREFIX}"
PROBLEMS = []
NEED = {T_EDGE_D: ["txn_date", "src_id", "src_kind", "dst_id", "dst_kind", "category", "is_self", "amount", "n_txn", "month"],
        T_MAP: ["cpty_id", "ent_l2"],
        T_PAIR: ["cust", "dir", "scope", "other", "amount", "o_fi", "o_class", "o_self", "month"],
        T_CUST: ["mdm_id", "is_tm"]}
for t, cols in NEED.items():
    if not table_exists(t):
        PROBLEMS.append({"problem": "table missing", "detail": t}); continue
    have = set(spark.table(t).columns)
    PROBLEMS += [{"problem": "column missing", "detail": f"{t}.{c}"} for c in cols if c not in have]
if not PROBLEMS:
    E_LOC, P_LOC = table_location(T_EDGE_D), table_location(T_PAIR)
    parts = lambda t: sorted(r[0].split("=", 1)[1] for r in spark.sql(f"SHOW PARTITIONS {t}").collect())
    E_PUB = [m for m in parts(T_EDGE_D) if exists(f"{E_LOC}/month={m}/_PKG_BUILD")]
    P_PUB = parts(T_PAIR)
    LAST = LAST_MONTH or (min(E_PUB[-1], P_PUB[-1]) if (E_PUB and P_PUB) else None)
    MONTHS = month_range(HISTORY_START, LAST) if LAST else []
    gaps_ = [m for m in MONTHS if m not in E_PUB or m not in P_PUB]
    if gaps_:
        PROBLEMS.append({"problem": "months missing in an input", "detail": ", ".join(gaps_)})
if PROBLEMS:
    out(sec(f"§1 stopped: {len(PROBLEMS)} problem(s)", tbl(pd.DataFrame(PROBLEMS))))
    raise AssertionError(f"§1: {len(PROBLEMS)} problem(s), see the table above")

RAIL = {"1": "ACH", "2": "ACH", "3": "ACH", "4": "WIRE", "5": "RTP_PRT", "6": "RTP_P2P", "7": "CHECK", "8": "DEBIT_CARD", "9": "PCARD"}
CFG = {"v": PER_VERSION, "start": HISTORY_START, "reg": [REG_WINDOW, REG_MAX_DAYS, MIN_PAYMENTS, REG_MIN, TOL_FRAC, TOL_MIN,
       SKIP_MULT, DAILY_STOP_BD, FIXED_PCT, MASS_STOP_MIN], "cad": CADENCES,
       "amt": [AMT_RECENT, METHODS, MEAN_CADENCES, MEAN_SIDES, list(BIAS_RANGE)],
       "cal": [SPREAD_BINS, AMT_BANDS, EDGE_DAYS, CALIB_LEVELS, CALIB_MONTHS, CALIB_MIN_MONTHS, CALIB_MIN_N, CALIB_PCT,
               list(INTERVAL)], "fc": FORECAST_MONTHS,
       "prune": PRUNE_IDLE_DAYS, "rail": RAIL, "hol": [h.isoformat() for h in HOLIDAYS], "map": T_MAP, "pair": T_PAIR}
SIG = md5(CFG)
def month_fp(m):
    return md5({"edge": read_text(f"{E_LOC}/month={m}/_PKG_BUILD"), "pair": ls_files(f"{P_LOC}/month={m}")})
marker = lambda m: read_json(f"{LOC['total']}/month={m}/_PKG_BUILD")

def month_why(m, mk, prev_build):
    """Why month m must be built (None = done): its marker must carry the current signature, the fingerprint of its
    inputs and the build id of the month before."""
    return ("REBUILD_FROM" if REBUILD_FROM and m >= REBUILD_FROM else
            "not built" if mk is None else
            "settings changed" if mk.get("sig") != SIG else
            "inputs changed" if mk.get("fp") != month_fp(m) else
            "month before rebuilt" if mk.get("prev_build") != prev_build else None)
def check_prev_state(m):
    """The first month to build needs the stream state of the month before, from the same build."""
    if m == MONTHS[0]:
        return
    pm = month_add(m, -1)
    sm, mk = read_json(f"{STATE}/month={pm}/_PKG_STATE.json"), marker(pm)
    if not (sm and mk and sm.get("build_id") == mk["build_id"]):
        raise AssertionError(f"the stream state of {pm} is gone (only the latest {KEEP_STATES} are kept): set "
                             f"REBUILD_FROM = HISTORY_START to rebuild from {HISTORY_START}")

# plan: walk the months; the first month not done (or REBUILD_FROM) starts the build, everything after it is rebuilt.
# §2 repeats this check month by month, so after a stop §2 can be run again on its own.
plan, prev_build, building = [], None, False
for m in MONTHS:
    mk = marker(m)
    why = month_why(m, mk, prev_build)
    building = building or why is not None
    if building:
        plan.append({"month": m, "action": "build", "why": why or "month before rebuilt"})
        prev_build = "?"
    else:
        plan.append({"month": m, "action": "done", "why": ""})
        prev_build = mk["build_id"]
TO_BUILD = [p["month"] for p in plan if p["action"] == "build"]
if TO_BUILD:
    check_prev_state(TO_BUILD[0])
_S1_OK = True
pl = pd.DataFrame(plan)
out(sec(f"§1 Inputs OK · signature {SIG} · months {MONTHS[0]} → {MONTHS[-1]}",
        tbl(pl.groupby(["action", "why"]).agg(months=("month", "count"), first=("month", "min"), last=("month", "max")).reset_index())),
    para(f"To build: {len(TO_BUILD)} month(s)" + (f", {TO_BUILD[0]} → {TO_BUILD[-1]}." if TO_BUILD else ".")
         + f" Output tables: {', '.join(TBL.values())}."))

## §2 Build

Per month, in order:
1. **Calibration** from the published months before it: per group the predictor, the bias correction and the
   percentiles of received ÷ corrected expected (streams), stop rates, and customer-level percentiles for this
   month's expectation and for next month's forecast.
2. **Payment days** of the month from `pkg_edge_daily` (counterparty ids → entity as in customer metrics, rail and raw
   id of the day's largest payment), with side, bank and class from the pair table.
3. **Stream step:** last month's state + this month's payments → features before and after the month, events,
   expectation for the month, forecast for the next months; written once to a work folder.
4. **Outputs:** state (pruned), stream rows, month and total tables, calibration rows; checked against the pair
   table (all dollars accounted for) before the marker is written.

In [ ]:
# ---- §2 Build ------------------------------------------------------------------------------------
assert globals().get("_S1_OK"), "§1 did not finish: fix the problem it shows, then run all cells again"

from contextlib import contextmanager
E = F.expr
K = ["cust", "dir", "scope", "other"]
PAY_T = "array<struct<d:int,b:int,a:double>>"
MON_T = "array<struct<mi:int,n:int,a:double>>"
STATE_SCHEMA = (f"cust string, dir string, scope string, other string, p {PAY_T}, mo {MON_T}, rail string, raw_id string, "
                f"bank string, o_class string, own boolean")
D0 = "to_date('1970-01-01')"
HOL_ARR = "array(" + ", ".join(f"date'{h.isoformat()}'" for h in HOLIDAYS) + ")"
_mis = range(month_idx("2015-01"), month_idx("2035-12") + 1)
BD_ARR = "array(" + ", ".join(f"named_struct('mi', {i}, 'bd', {business_days(f'{i // 12:04d}-{i % 12 + 1:02d}')})" for i in _mis
                              if month_idx(HISTORY_START) - 30 <= i <= month_idx(MONTHS[-1]) + 6) + ")"
sk = float(SKIP_MULT)
PROJ_STEPS = ((FORECAST_MONTHS + 1) * 31 + 5) // 6 + 1   # projected dates per stream: enough for a weekly stream (6-day
                                                         # gap) over the month and the FORECAST_MONTHS after it
AMT_OF = {"med3": "med3", "mean": "mean", "med": "meda", "last": "last_a"}   # predictor → per-payment amount feature
assert set(METHODS) <= set(AMT_OF), f"METHODS must come from {list(AMT_OF)}"
KEYCOL = {"dir": "dir", "side": "side", "cadence": "cad", "rail": "rail", "amt_band": "band", "spread_bin": "sbin", "edge": "edge"}
CAL_KEYS = list(KEYCOL)
jcol = lambda k, s: k if k in ("dir", "side") else f"{KEYCOL[k]}{s}"     # calibration key → stream column (suffix _p / _n)
key_type = lambda k: "boolean" if k == "edge" else "string"

@contextmanager
def many_partitions(n):
    keys = ("spark.sql.shuffle.partitions", "spark.sql.adaptive.coalescePartitions.enabled")
    old = {k: spark.conf.get(k, None) for k in keys}
    spark.conf.set(keys[0], str(n)); spark.conf.set(keys[1], "false")
    try:
        yield
    finally:
        for k, v in old.items():
            spark.conf.unset(k) if v is None else spark.conf.set(k, v)

def _gaps(x):
    return (f"if(size({x}) >= 2, zip_with(slice({x}, 2, size({x}) - 1), slice({x}, 1, size({x}) - 1), (u, v) -> u - v), "
            f"cast(array() as array<int>))")
def _nth(x, q):
    """q-quantile of an array (lower element of the sorted array, as v4); NULL when empty."""
    return f"if(size({x}) = 0, null, element_at(array_sort({x}), cast(floor((size({x}) - 1) * {q}) as int) + 1))"
def shift_bd(x):
    """A date on a weekend moves to Monday, then a Fed holiday to the next day (Friday holiday → Monday)."""
    s = f"case dayofweek({x}) when 7 then date_add({x}, 2) when 1 then date_add({x}, 1) else {x} end"
    return f"if(array_contains({HOL_ARR}, {s}), date_add({s}, if(dayofweek({s}) = 6, 3, 1)), {s})"
def spread_bin(c):
    """Spread (MAD / median) in SPREAD_BINS: "0" = fixed amount, then "0–0.005", ..., "> 0.25"; NULL stays NULL."""
    e, lo = F.when(c == 0, F.lit("0")), 0.0
    for hi in SPREAD_BINS[1:]:
        e = e.when(c <= hi, F.lit(f"{lo:g}–{hi:g}")); lo = hi
    return e.when(c > lo, F.lit(f"> {lo:g}"))
def _usd(v):
    return f"{v / 1e6:g}M" if v >= 1e6 else f"{v / 1e3:g}k" if v >= 1e3 else f"{v:g}"
def amt_band(c):
    """Median payment in AMT_BANDS: "< 100", "100–1k", ..., "≥ 1M"; NULL stays NULL."""
    e = F.when(c < AMT_BANDS[0], F.lit(f"< {_usd(AMT_BANDS[0])}"))
    for lo, hi in zip(AMT_BANDS, AMT_BANDS[1:]):
        e = e.when(c < hi, F.lit(f"{_usd(lo)}–{_usd(hi)}"))
    return e.when(c >= AMT_BANDS[-1], F.lit(f"≥ {_usd(AMT_BANDS[-1])}"))
def edge_of(s, m):
    """A projected payment date within EDGE_DAYS of the start or the end of month m: a shift of a few days moves it into
    the month next to it. Daily streams: false."""
    def near(d):
        return (f"(x >= date'{(d - dt.timedelta(days=EDGE_DAYS)).isoformat()}' and "
                f"x < date'{(d + dt.timedelta(days=EDGE_DAYS)).isoformat()}')")
    a, b = month_first(m), month_first(month_add(m, 1))
    return F.coalesce(~F.col(f"daily{s}") & E(f"exists(pd{s}, x -> {near(a)} or {near(b)})"), F.lit(False))

def feats(J, pc, mc, s, mx):
    """Stream features from a payment-day array pc and a month array mc (suffix s); mx = month index the
    projection starts from (the daily fill rate uses the 3 months before it)."""
    J = (J.withColumn(f"dd{s}", E(f"transform({pc}, x -> x.d)")).withColumn(f"bb{s}", E(f"transform({pc}, x -> x.b)"))
          .withColumn(f"aa{s}", E(f"transform({pc}, x -> x.a)")))
    J = J.withColumn(f"g{s}", E(_gaps(f"dd{s}"))).withColumn(f"gb{s}", E(_gaps(f"bb{s}")))
    J = (J.withColumn(f"med{s}", E(_nth(f"g{s}", 0.5))).withColumn(f"medb{s}", E(_nth(f"gb{s}", 0.5)))
          .withColumn(f"daily{s}", F.coalesce(F.col(f"medb{s}") <= 1, F.lit(False)))
          .withColumn(f"tol{s}", F.greatest(F.lit(float(TOL_MIN)), F.col(f"med{s}") * float(TOL_FRAC))))
    J = J.withColumn(f"ont{s}", E(f"if(size(g{s}) = 0, null, if(daily{s}, size(filter(gb{s}, x -> x <= 1)) / size(gb{s}), "
                                  f"size(filter(g{s}, x -> abs(x - med{s}) <= tol{s})) / size(g{s})))"))
    cad = F.lit(None).cast("string")
    for g, lo, hi in CADENCES:
        cad = F.when(F.col(f"med{s}").between(lo, hi), F.lit(g)).otherwise(cad)
    cad = F.when(F.col(f"daily{s}"), F.lit("daily")).otherwise(cad)
    J = (J.withColumn(f"npay{s}", E(f"aggregate({mc}, 0, (acc, x) -> acc + x.n)"))
          .withColumn(f"cad{s}", cad)
          .withColumn(f"reg{s}", F.coalesce((F.col(f"npay{s}") >= MIN_PAYMENTS) & F.col(f"cad{s}").isNotNull()
                                            & (F.col(f"ont{s}") >= REG_MIN), F.lit(False)))
          .withColumn(f"meda{s}", E(_nth(f"aa{s}", 0.5))).withColumn(f"p10a{s}", E(_nth(f"aa{s}", 0.1)))
          .withColumn(f"p90a{s}", E(_nth(f"aa{s}", 0.9)))
          .withColumn(f"med3{s}", E(_nth(f"slice(aa{s}, greatest(size(aa{s}) - {AMT_RECENT - 1}, 1), {AMT_RECENT})", 0.5)))
          .withColumn(f"mean{s}", E(f"if(size(aa{s}) = 0, null, aggregate(aa{s}, cast(0 as double), (t, x) -> t + x) / size(aa{s}))"))
          .withColumn(f"sd{s}", E(f"if(size(aa{s}) = 0, null, sqrt(aggregate(aa{s}, cast(0 as double), "
                                  f"(t, x) -> t + (x - mean{s}) * (x - mean{s})) / size(aa{s})))"))
          .withColumn(f"mad{s}", E(_nth(f"transform(aa{s}, x -> abs(x - meda{s}))", 0.5)))
          .withColumn(f"spread{s}", F.when(F.col(f"meda{s}") > 0, F.col(f"mad{s}") / F.col(f"meda{s}")))
          .withColumn(f"p10g{s}", E(f"if(daily{s}, {_nth(f'gb{s}', 0.1)}, {_nth(f'g{s}', 0.1)})"))
          .withColumn(f"p90g{s}", E(f"if(daily{s}, {_nth(f'gb{s}', 0.9)}, {_nth(f'g{s}', 0.9)})"))
          .withColumn(f"dom{s}", E(_nth(f"slice(transform(dd{s}, x -> dayofmonth(date_add({D0}, x))), "
                                        f"greatest(size(dd{s}) - 2, 1), 3)", 0.5)))
          .withColumn(f"cm{s}", F.when(F.col(f"cad{s}") == "monthly", 1)
                                 .when((F.col(f"cad{s}") == "quarterly") & (F.col(f"med{s}") < 70), 2)
                                 .when(F.col(f"cad{s}") == "quarterly", 3))
          .withColumn(f"m0{s}", E(f"array_min(transform({mc}, x -> x.mi))"))
          .withColumn(f"fill{s}", E(f"least(1.0, aggregate(filter({mc}, x -> x.mi between {mx - 3} and {mx - 1}), 0, (t, x) -> t + x.n) / "
                                    f"nullif(aggregate(filter({BD_ARR}, y -> y.mi between greatest({mx - 3}, m0{s}) and {mx - 1}), 0, "
                                    f"(t, y) -> t + y.bd), 0))"))
          .withColumn(f"lastd{s}", E(f"element_at(dd{s}, -1)")).withColumn(f"lastb{s}", E(f"element_at(bb{s}, -1)"))
          .withColumn(f"last_a{s}", E(f"element_at(aa{s}, -1)"))
          .withColumn(f"band{s}", F.coalesce(amt_band(F.col(f"meda{s}")), F.lit("?")))
          .withColumn(f"sbin{s}", F.coalesce(spread_bin(F.col(f"spread{s}")), F.lit("?"))))
    # projected payment dates after the last payment (PROJ_STEPS steps)
    ld = f"date_add({D0}, lastd{s})"
    cal = (f"make_date(year(add_months({ld}, k * cm{s})), month(add_months({ld}, k * cm{s})), "
           f"least(dom{s}, day(last_day(add_months({ld}, k * cm{s})))))")
    raw = f"transform(sequence(1, {PROJ_STEPS}), k -> if(cm{s} is not null, {cal}, date_add({ld}, cast(round(k * med{s}) as int))))"
    return J.withColumn(f"pd{s}", E(f"if(lastd{s} is null or daily{s} or med{s} is null, null, transform({raw}, x -> {shift_bd('x')}))"))

def n_in(s, m):
    """Payments expected in month m from the features with suffix s: projected dates in the month, daily = business
    days × fill rate."""
    a, b = month_first(m).isoformat(), month_last(m).isoformat()
    return (F.when(F.col(f"daily{s}"), F.lit(float(business_days(m))) * F.coalesce(F.col(f"fill{s}"), F.lit(1.0)))
             .otherwise(E(f"cast(size(filter(pd{s}, x -> x >= date'{a}' and x <= date'{b}')) as double)")))

def lk_table():
    return (spark.table(T_MAP).groupBy("cpty_id").agg(F.min("ent_l2").alias("ent")))

def pay_days(m, lk):
    dd = spark.table(T_EDGE_D).where((F.col("month") == m) & ~F.coalesce(F.col("is_self"), F.lit(False)))
    keep = ["txn_date", "amount", "n_txn", "category"]
    out_ = dd.where(F.col("src_kind") == "CUST").select(F.col("src_id").alias("cust"), F.lit("out").alias("dir"),
                                                       F.col("dst_id").alias("oid"), F.col("dst_kind").alias("okind"), *keep)
    in_ = dd.where(F.col("dst_kind") == "CUST").select(F.col("dst_id").alias("cust"), F.lit("in").alias("dir"),
                                                      F.col("src_id").alias("oid"), F.col("src_kind").alias("okind"), *keep)
    p = (out_.unionByName(in_).withColumn("scope", F.when(F.col("okind") == "CUST", "int").otherwise("ext"))
             .withColumn("rail", F.create_map(*[F.lit(x) for kv in RAIL.items() for x in kv])[F.substring("category", 1, 1)]))
    ext = (p.where(F.col("scope") == "ext").join(lk.select(F.col("cpty_id").alias("oid"), "ent"), "oid", "left")
            .withColumn("other", F.coalesce("ent", "oid")).drop("ent"))
    intr = p.where(F.col("scope") == "int").withColumn("other", F.col("oid"))
    a = F.col("amount").cast("double")
    return (ext.unionByName(intr).groupBy(*K, "txn_date")
               .agg(F.sum(a).alias("a"), F.sum("n_txn").cast("long").alias("n"),
                    F.max(F.struct(a.alias("x"), F.col("rail"), F.col("oid"))).alias("_m")))

def load_state(m, prev_build):
    pm = month_add(m, -1)
    if m == MONTHS[0] or pm < HISTORY_START:
        return spark.createDataFrame([], STATE_SCHEMA)
    sm = read_json(f"{STATE}/month={pm}/_PKG_STATE.json")
    if not (sm and sm.get("build_id") == prev_build):
        raise AssertionError(f"state of {pm} missing or from another build: set REBUILD_FROM = HISTORY_START")
    return spark.read.parquet(f"{STATE}/month={pm}")

def nbucket(c):
    return F.when(c <= 1, "1").when(c <= 3, "2–3").when(c <= 9, "4–9").otherwise("10+")

def _native(v):
    """pandas / numpy value → Python value for createDataFrame (NaN → None)."""
    if v is None or (isinstance(v, float) and v != v):
        return None
    try:
        if pd.isna(v):
            return None
    except (TypeError, ValueError):
        pass
    return v.item() if hasattr(v, "item") else v

def calibration(m, work):
    """For month m, from the published months before it (CALIB_MONTHS): per calibration group the predictor, its bias
    correction and the percentiles of received ÷ corrected expected (stream level); stop rates; customer-level
    percentiles for this month's expectation and for next month's forecast. Returns (stream groups, stop rates,
    customer expectation, customer forecast, number of months used)."""
    prior = [x for x in month_range(month_add(m, -CALIB_MONTHS), month_add(m, -1))
             if x >= MONTHS[0] and exists(f"{LOC['total']}/month={x}/_PKG_BUILD")]
    if len(prior) < CALIB_MIN_MONTHS:
        return None, None, None, None, len(prior)
    smp = E(f"pmod(xxhash64(cust), 100) < {int(CALIB_PCT)}")
    lo, hi = INTERVAL
    # the sample of due streams, written once and read by every pass below
    (spark.read.parquet(*[f"{LOC['stream']}/month={x}" for x in prior]).where(smp & F.col("due"))
          .select("dir", "side", F.col("cadence_p").alias("cadence"), F.col("rail_p").alias("rail"),
                  F.col("amt_band_p").alias("amt_band"), F.col("spread_bin_p").alias("spread_bin"), F.col("edge_p").alias("edge"),
                  "stopped", "amt", *[F.coalesce(F.col(f"exp_{k}"), F.lit(0.0)).alias(f"exp_{k}") for k in METHODS])
          .write.mode("overwrite").parquet(f"{work}/cal_base"))
    base = spark.read.parquet(f"{work}/cal_base")
    ok = base.where(~F.col("stopped"))
    clip = lambda c: F.least(F.greatest(c, F.lit(float(BIAS_RANGE[0]))), F.lit(float(BIAS_RANGE[1])))
    groups = []
    for tier, keys in enumerate(CALIB_LEVELS, 1):
        # 1 bias of every predictor: received ÷ expected of the group
        g = (ok.groupBy(*keys).agg(F.count("*").alias("n"), F.sum("amt").alias("_sa"), *[F.sum(f"exp_{k}").alias(f"_se_{k}") for k in METHODS])
               .where((F.col("n") >= CALIB_MIN_N) & (F.col("_sa") > 0)))
        g = g.select(*keys, "n", "_sa", *[F.when(F.col(f"_se_{k}") > 0, clip(F.col("_sa") / F.col(f"_se_{k}"))).alias(f"_b_{k}") for k in METHODS])
        g = g.toPandas()
        if not len(g):
            continue
        gs = spark.createDataFrame([tuple(_native(v) for v in r) for r in g[keys + ["_sa"] + [f"_b_{k}" for k in METHODS]].itertuples(index=False)],
                                   ", ".join(f"`{k}` {key_type(k)}" for k in keys) + ", _sa double, "
                                   + ", ".join(f"_b_{k} double" for k in METHODS))
        # 2 dollar error of every predictor after its bias correction; the predictor with the lowest error
        er = (ok.join(F.broadcast(gs), keys)
                .groupBy(*keys).agg(*[(F.sum(F.abs(F.col("amt") - F.col(f"_b_{k}") * F.col(f"exp_{k}"))) / F.first("_sa")).alias(f"err_{k}")
                                      for k in METHODS]).toPandas())
        g = g.merge(er, on=keys)
        errs = g[[f"err_{k}" for k in METHODS]].astype(float)
        g = g[errs.notna().any(axis=1)].copy()
        if not len(g):
            continue
        errs = g[[f"err_{k}" for k in METHODS]].astype(float).fillna(float("inf"))
        g["method"] = [METHODS[i] for i in errs.values.argmin(axis=1)]          # argmin keeps the first on ties
        g["bias"] = [float(r[f"_b_{r['method']}"]) for _, r in g.iterrows()]
        # 3 percentiles of received ÷ (bias × expected of the chosen predictor)
        ch = spark.createDataFrame([tuple(_native(v) for v in r) for r in g[keys + ["method", "bias"]].itertuples(index=False)],
                                   ", ".join(f"`{k}` {key_type(k)}" for k in keys) + ", method string, bias double")
        e_ch = F.coalesce(*[F.when(F.col("method") == k, F.col(f"exp_{k}")) for k in METHODS]) * F.col("bias")
        q = (ok.join(F.broadcast(ch), keys).withColumn("_e", e_ch).where(F.col("_e") > 0)
               .groupBy(*keys).agg(F.percentile_approx(F.col("amt") / F.col("_e"), [lo, 0.5, hi], 1000).alias("_q"))
               .select(*keys, F.col("_q")[0].alias("q_lo"), F.col("_q")[1].alias("q_mid"), F.col("_q")[2].alias("q_hi")).toPandas())
        g = g.merge(q, on=keys, how="left")
        g["tier"] = tier
        groups.append(g.drop(columns=["_sa"] + [f"_b_{k}" for k in METHODS]))
    stream_cal = pd.concat(groups, ignore_index=True) if groups else None
    # stop rates: share of due streams that stopped in a month, side × cadence × rail (and side × cadence)
    hz = (base.groupBy("side", "cadence", "rail").agg(F.count("*").alias("n"), F.avg(F.col("stopped").cast("double")).alias("hazard"))
              .unionByName(base.groupBy("side", "cadence").agg(F.count("*").alias("n"), F.avg(F.col("stopped").cast("double")).alias("hazard"))
                               .withColumn("rail", F.lit("*")))
              .toPandas())
    # customer level: this month's expectation (amt_due ÷ exp_amt by number of due streams) and next month's forecast
    # (what came ÷ fc1_amt of the month before, by number of live streams)
    tb = spark.read.option("basePath", LOC["total"]).parquet(*[f"{LOC['total']}/month={x}" for x in prior]).where(smp)
    qc = lambda df, r: (df.groupBy("dir", "side", "nb").agg(F.count("*").alias("n"), F.percentile_approx(r, [lo, 0.5, hi], 1000).alias("_q"))
                          .select("dir", "side", "nb", "n", F.col("_q")[0].alias("q_lo"), F.col("_q")[1].alias("q_mid"),
                                  F.col("_q")[2].alias("q_hi")).toPandas())
    cust_exp = qc(tb.where(F.col("exp_amt") > 0).withColumn("nb", nbucket(F.col("n_due"))), F.col("amt_due") / F.col("exp_amt"))
    nxt = {month_add(x, -1): x for x in prior if month_add(x, -1) in prior}
    cust_fc1 = None
    if nxt:
        fa = (tb.where(F.col("month").isin(list(nxt)) & (F.col("fc1_amt") > 0))
                .select("cust", "dir", "side", F.create_map(*[F.lit(v) for kv in nxt.items() for v in kv])[F.col("month")].alias("_cm"),
                        "fc1_amt", nbucket(F.col("n_live")).alias("nb")))
        ca = tb.select("cust", "dir", "side", F.col("month").alias("_cm"), "amt_due")
        cust_fc1 = qc(fa.join(ca, ["cust", "dir", "side", "_cm"], "left").fillna({"amt_due": 0.0}), F.col("amt_due") / F.col("fc1_amt"))
    return stream_cal, hz, cust_exp, cust_fc1, len(prior)

def apply_cal(J, cal, s):
    """Predictor, bias and interval percentiles per stream (suffix s) from the first CALIB_LEVELS group that covers it;
    without one, the default predictor (mean for MEAN_CADENCES / MEAN_SIDES, else med3), no bias correction, no interval.
    Adds meth{s}, bias{s}, apay{s} (dollars per payment), q_lo{s}, q_hi{s}, lvl{s}."""
    picks = []
    if cal is not None:
        for tier, keys in enumerate(CALIB_LEVELS, 1):
            g = cal[cal.tier == tier]
            if not len(g):
                continue
            cols = [jcol(k, s) for k in keys]
            t = spark.createDataFrame([tuple(_native(v) for v in r) for r in g[keys + ["method", "bias", "q_lo", "q_hi"]].itertuples(index=False)],
                                      ", ".join(f"`{c}` {key_type(k)}" for k, c in zip(keys, cols))
                                      + f", _m{tier} string, _b{tier} double, _lo{tier} double, _hi{tier} double")
            J = J.join(F.broadcast(t), cols, "left")
            picks.append(F.when(F.col(f"_m{tier}").isNotNull(),
                                F.struct(F.col(f"_m{tier}").alias("m"), F.col(f"_b{tier}").alias("b"), F.col(f"_lo{tier}").alias("lo"),
                                         F.col(f"_hi{tier}").alias("hi"), F.lit(tier).alias("l"))))
    cs = "struct<m:string,b:double,lo:double,hi:double,l:int>"
    J = J.withColumn("_c", F.coalesce(*picks) if len(picks) > 1 else picks[0] if picks else F.lit(None).cast(cs))
    J = J.drop(*[f"_{x}{t}" for t in range(1, len(CALIB_LEVELS) + 1) for x in ("m", "b", "lo", "hi")])
    dm = F.when(F.col(f"cad{s}").isin(MEAN_CADENCES) | F.col("side").isin(MEAN_SIDES), F.lit("mean")).otherwise(F.lit("med3"))
    meth = F.coalesce(F.col("_c.m"), dm)
    amt = F.lit(None).cast("double")
    for k, f_ in AMT_OF.items():
        amt = F.when(meth == k, F.col(f"{f_}{s}")).otherwise(amt)
    return (J.withColumn(f"meth{s}", meth).withColumn(f"bias{s}", F.coalesce(F.col("_c.b"), F.lit(1.0)))
             .withColumn(f"apay{s}", amt * F.col(f"bias{s}"))
             .withColumn(f"q_lo{s}", F.col("_c.lo")).withColumn(f"q_hi{s}", F.col("_c.hi")).withColumn(f"lvl{s}", F.col("_c.l"))
             .drop("_c"))

def build_month(m, prev_build, build_id):
    t0 = time.time(); steps = {}
    mi = month_idx(m)
    WS_D = (month_first(month_add(m, -(REG_WINDOW - 1))) - EPOCH).days
    end, pend = month_last(m), month_first(m) - dt.timedelta(days=1)
    END_D, END_B, PEND_D, PEND_B = (end - EPOCH).days, bidx_py(end), (pend - EPOCH).days, bidx_py(pend)
    work = f"{WORK}/month={m}"
    delete(work)
    # 1 calibration
    scal, hz, ccal, fcal, n_prior = calibration(m, work)
    steps["calib"] = round((time.time() - t0) / 60, 2)
    # 2 payment days and pair attributes
    pdays = pay_days(m, LK)
    cur = (pdays.withColumn("d", F.datediff("txn_date", E(D0))).withColumn("b", bidx(F.col("txn_date")).cast("int"))
                .groupBy(*K).agg(F.array_sort(F.collect_list(F.struct("d", "b", "a"))).alias("cp"),
                                 F.sum("a").alias("c_amt"), F.sum("n").alias("c_ntxn"),
                                 F.max(F.struct("d", F.col("_m.rail").alias("rail"), F.col("_m.oid").alias("raw"))).alias("_l")))
    pa = (spark.table(T_PAIR).where((F.col("month") == m) & F.col("scope").isin("int", "ext"))
               .select(*K, F.col("o_fi").alias("c_bank"), F.col("o_class").alias("c_class"), F.col("o_self").alias("c_own")))
    cur = cur.join(pa, K, "left")
    with many_partitions(PER_PARTITIONS):
        J = (load_state(m, prev_build).join(cur, K, "full")
               .withColumn("p", F.coalesce("p", E(f"cast(array() as {PAY_T})")))
               .withColumn("mo", F.coalesce("mo", E(f"cast(array() as {MON_T})")))
               .withColumn("cp", F.coalesce("cp", E(f"cast(array() as {PAY_T})")))
               .withColumn("c_amt", F.coalesce("c_amt", F.lit(0.0))).withColumn("c_ntxn", F.coalesce("c_ntxn", F.lit(0)).cast("long"))
               .withColumn("act", F.size("cp") > 0)
               .withColumn("rail_p", F.coalesce(F.col("rail"), F.lit("?")))          # rail before the month (calibration key)
               .withColumn("rail", F.coalesce(F.col("_l.rail"), "rail")).withColumn("raw_id", F.coalesce(F.col("_l.raw"), "raw_id"))
               .withColumn("bank", F.coalesce("c_bank", "bank")).withColumn("o_class", F.coalesce("c_class", "o_class"))
               .withColumn("own", F.coalesce("c_own", "own", F.lit(False)))
               .withColumn("side", F.when(F.col("scope") == "int", "int").when(F.col("own"), "own").otherwise("ext"))
               .drop("_l", "c_bank", "c_class", "c_own"))
        # history before the month (suffix _p): what was expected
        J = feats(J, "p", "mo", "_p", mi).withColumn("edge_p", edge_of("_p", m))
        J = (J.withColumn("stop_p", F.coalesce(F.col("reg_p") & F.when(F.col("daily_p"), F.lit(PEND_B) - F.col("lastb_p") >= DAILY_STOP_BD)
                                                                   .otherwise(F.lit(PEND_D) - F.col("lastd_p") >= sk * F.col("med_p")), F.lit(False)))
              .withColumn("due", F.col("reg_p") & ~F.col("stop_p"))
              .withColumn("n_exp", F.when(F.col("due"), n_in("_p", m))))
        J = apply_cal(J, scal, "_p")
        J = (J.withColumn("exp_amt", F.col("n_exp") * F.col("apay_p"))
              .withColumn("exp_med3", F.col("n_exp") * F.col("med3_p")).withColumn("exp_mean", F.col("n_exp") * F.col("mean_p"))
              .withColumn("exp_med", F.col("n_exp") * F.col("meda_p")).withColumn("exp_last", F.col("n_exp") * F.col("last_a_p")))
        # this month's payments, each with its gap to the payment before it
        J = (J.withColumn("nd", E("if(size(p) = 0, transform(cp, x -> x.d), concat(array(lastd_p), transform(cp, x -> x.d)))"))
              .withColumn("nb", E("if(size(p) = 0, transform(cp, x -> x.b), concat(array(lastb_p), transform(cp, x -> x.b)))"))
              .withColumn("gn", E(_gaps("nd"))).withColumn("gbn", E(_gaps("nb"))))
        nondaily = F.col("reg_p") & ~F.col("daily_p")
        J = (J.withColumn("late", F.coalesce(nondaily & E(f"exists(gn, x -> x > med_p + tol_p and x < {sk} * med_p)"), F.lit(False)))
              .withColumn("early", F.coalesce(F.col("due") & ~F.col("daily_p") & E("exists(gn, x -> x < med_p - tol_p)"), F.lit(False)))
              .withColumn("skipped", F.coalesce(F.col("reg_p") & F.when(F.col("daily_p"), E("exists(gbn, x -> x >= 2)"))
                                                                   .otherwise(E(f"exists(gn, x -> x >= {sk} * med_p)")), F.lit(False)))
              .withColumn("days_late", F.when(F.col("late") | F.col("skipped"),
                                              F.when(F.col("daily_p"), E("array_max(gbn) - 1"))
                                               .otherwise(E("cast(round(array_max(transform(gn, x -> x - med_p))) as int)"))))
              .withColumn("amt_low", F.coalesce(F.col("due") & ~F.col("daily_p") & (F.col("npay_p") >= MIN_PAYMENTS)
                                                & E(f"exists(transform(cp, x -> x.a), x -> x < least(2 * p10a_p - meda_p, meda_p * {1 - FIXED_PCT}))"),
                                                F.lit(False)))
              .withColumn("amt_high", F.coalesce(F.col("due") & ~F.col("daily_p") & (F.col("npay_p") >= MIN_PAYMENTS)
                                                 & E(f"exists(transform(cp, x -> x.a), x -> x > greatest(2 * p90a_p - meda_p, meda_p * {1 + FIXED_PCT}))"),
                                                 F.lit(False)))
              .withColumn("resumed", F.coalesce(F.col("reg_p") & F.col("stop_p") & F.col("act"), F.lit(False))))
        # history after the month (suffix _n): the state and the forecast
        J = (J.withColumn("pf", E(f"filter(concat(p, cp), x -> x.d >= {WS_D})"))
              .withColumn("p2", E(f"slice(pf, greatest(size(pf) - {REG_MAX_DAYS} + 1, 1), {REG_MAX_DAYS})"))
              .withColumn("mo2", E(f"filter(concat(mo, if(act, array(named_struct('mi', {mi}, 'n', size(cp), 'a', c_amt)), "
                                   f"cast(array() as {MON_T}))), x -> x.mi >= {mi - REG_WINDOW + 1})")))
        J = feats(J, "p2", "mo2", "_n", mi + 1)
        J = (J.withColumn("idle_d", F.lit(END_D) - F.col("lastd_n")).withColumn("idle_b", F.lit(END_B) - F.col("lastb_n"))
              .withColumn("stop_n", F.coalesce(F.col("reg_n") & F.when(F.col("daily_n"), F.col("idle_b") >= DAILY_STOP_BD)
                                                               .otherwise(F.col("idle_d") >= sk * F.col("med_n")), F.lit(False)))
              .withColumn("live", F.col("reg_n") & ~F.col("stop_n"))
              .withColumn("stopped", F.coalesce(F.col("due") & F.when(F.col("daily_p"), F.col("idle_b") >= DAILY_STOP_BD)
                                                                  .otherwise(F.col("idle_d") >= sk * F.col("med_p")), F.lit(False)))
              .withColumn("overdue", F.coalesce(F.col("due") & ~F.col("stopped")
                                                & F.when(F.col("daily_p"), F.col("idle_b") >= 2)
                                                   .otherwise(F.col("idle_d") > F.col("med_p") + F.col("tol_p")), F.lit(False)))
              .withColumn("became_regular", F.col("reg_n") & ~F.col("reg_p"))
              .withColumn("shortfall", F.when(F.col("due"), F.greatest(F.col("exp_amt") - F.col("c_amt"), F.lit(0.0))))
              .withColumn("s12", E(f"aggregate(filter(mo2, x -> x.mi >= {mi - 11}), cast(0 as double), (t, x) -> t + x.a)"))
              .withColumn("rail_n", F.coalesce(F.col("rail"), F.lit("?"))).withColumn("edge_n", edge_of("_n", month_add(m, 1))))
        for k in range(1, FORECAST_MONTHS + 1):
            J = J.withColumn(f"fn{k}", F.when(F.col("live"), n_in("_n", month_add(m, k))))
        J = (J.withColumn("next_dt", F.when(F.col("live"), F.when(F.col("daily_n"), E(shift_bd(f"date_add({D0}, lastd_n + 1)")))
                                                           .otherwise(E("element_at(pd_n, 1)"))))
              .withColumn("next_lo_dt", F.when(F.col("live") & ~F.col("daily_n"), E(f"date_add({D0}, lastd_n + p10g_n)")))
              .withColumn("next_hi_dt", F.when(F.col("live") & ~F.col("daily_n"), E(f"date_add({D0}, lastd_n + p90g_n)"))))
        # predictor, bias and interval for the months after (keys at the end of the month); stop rates (side × cadence
        # × rail, else side × cadence)
        J = apply_cal(J, scal, "_n")
        if hz is None:
            J = J.withColumn("hazard", F.lit(None).cast("double"))
        else:
            hf = hz[(hz.rail != "*") & (hz.n >= CALIB_MIN_N)][["side", "cadence", "rail", "hazard"]]
            hp = hz[(hz.rail == "*") & (hz.n >= CALIB_MIN_N)][["side", "cadence", "hazard"]]
            hf = spark.createDataFrame([tuple(_native(v) for v in r) for r in hf.itertuples(index=False)], "side string, cad_n string, rail_n string, hf double")
            hp = spark.createDataFrame([tuple(_native(v) for v in r) for r in hp.itertuples(index=False)], "side string, cad_n string, hp double")
            J = (J.join(F.broadcast(hf), ["side", "cad_n", "rail_n"], "left").join(F.broadcast(hp), ["side", "cad_n"], "left")
                  .withColumn("hazard", F.coalesce("hf", "hp")).drop("hf", "hp"))
        pc = 1 - F.coalesce(F.col("hazard"), F.lit(0.0))
        fc3 = sum(F.col(f"fn{k}") * F.col("apay_n") * F.pow(pc, k) for k in range(1, FORECAST_MONTHS + 1))
        J = (J.withColumn("exp_lo", F.col("exp_amt") * F.col("q_lo_p")).withColumn("exp_hi", F.col("exp_amt") * F.col("q_hi_p"))
              .withColumn("fc1_amt", F.col("fn1") * F.col("apay_n"))
              .withColumn("fc1_lo", F.col("fc1_amt") * F.col("q_lo_n")).withColumn("fc1_hi", F.col("fc1_amt") * F.col("q_hi_n"))
              .withColumn("p_continue", F.when(F.col("live"), 1 - F.col("hazard")))
              .withColumn("fc3_amt", F.when(F.col("live"), fc3)))
        # payer-wide stops (incoming): the payer stopped ≥ MASS_STOP_MIN customers this month
        pw = (J.where(F.col("stopped") & (F.col("dir") == "in")).groupBy("other").agg(F.count("*").alias("_pw"))
               .where(F.col("_pw") >= MASS_STOP_MIN).select("other", F.lit(True).alias("payer_wide")))
        J = J.join(F.broadcast(pw), "other", "left").withColumn("payer_wide", F.coalesce(F.col("payer_wide") & F.col("stopped"), F.lit(False)))
        keep = (K + ["side", "p2", "mo2", "rail", "rail_p", "raw_id", "bank", "o_class", "own", "act", "c_amt", "c_ntxn", "reg_p",
                     "reg_n", "due", "live", "stopped", "overdue", "late", "early", "skipped", "days_late", "amt_low", "amt_high",
                     "resumed", "became_regular", "payer_wide", "cad_p", "cad_n", "sbin_p", "band_p", "edge_p", "n_exp", "exp_amt",
                     "exp_lo", "exp_hi", "meth_p", "bias_p", "lvl_p", "exp_med3", "exp_mean", "exp_med", "exp_last", "shortfall",
                     "npay_n", "lastd_n", "idle_d", "med_n", "p10g_n", "p90g_n", "ont_n", "meda_n", "med3_n", "last_a_n", "mean_n",
                     "sd_n", "mad_n", "spread_n", "p10a_n", "p90a_n", "dom_n", "fill_n", "s12", "next_dt", "next_lo_dt",
                     "next_hi_dt", "fn1", "fc1_amt", "fc1_lo", "fc1_hi", "meth_n", "bias_n", "p_continue", "fc3_amt", "first_d"])
        J = J.withColumn("first_d", E("array_min(transform(p2, x -> x.d))")).select(*keep)
        J.write.mode("overwrite").parquet(f"{work}/j")
    steps["streams"] = round((time.time() - t0) / 60 - sum(steps.values()), 2)
    J = spark.read.parquet(f"{work}/j")
    # 4a state, pruned: streams with too few payments that went quiet can no longer become regular
    st = J.where((F.size("mo2") > 0) & ((F.col("npay_n") >= MIN_PAYMENTS) | (F.col("idle_d") <= PRUNE_IDLE_DAYS)))
    (st.select(*K, F.col("p2").alias("p"), F.col("mo2").alias("mo"), "rail", "raw_id", "bank", "o_class", "own")
       .write.mode("overwrite").parquet(f"{STATE}/month={m}"))
    write_json(f"{STATE}/month={m}/_PKG_STATE.json", {"build_id": build_id, "sig": SIG})
    # 4b stream rows
    d_ = lambda c: E(f"date_add({D0}, {c})")
    tot12 = J.groupBy("cust", "dir").agg(F.sum("s12").alias("_t12"))
    S = (J.where(F.col("due") | F.col("live") | F.col("resumed")).join(tot12, ["cust", "dir"], "left")
          .select(*K, "side", F.col("raw_id"), "bank", "o_class", "rail",
                  F.coalesce(F.when(F.col("live"), F.col("cad_n")), F.col("cad_p")).alias("cadence"),
                  F.col("cad_p").alias("cadence_p"), "rail_p", F.col("band_p").alias("amt_band_p"), F.col("sbin_p").alias("spread_bin_p"),
                  "edge_p",
                  F.when(F.col("stopped"), "stopped").when(F.col("overdue"), "overdue").when(F.col("live"), "live")
                   .when(F.col("resumed"), "resumed").otherwise("not regular").alias("status"),
                  "due", "live", F.col("npay_n").alias("n_pay"), d_("first_d").alias("first_dt"), d_("lastd_n").alias("last_dt"),
                  F.col("med_n").alias("med_gap"), F.col("p10g_n").alias("gap_p10"), F.col("p90g_n").alias("gap_p90"),
                  F.col("ont_n").alias("ontime_share"), F.col("dom_n").alias("pay_dom"), F.col("fill_n").alias("daily_fill"),
                  F.col("meda_n").alias("amt_med"), F.col("med3_n").alias("amt_med3"), F.col("last_a_n").alias("amt_last"),
                  F.col("mean_n").alias("amt_mean"), F.col("sd_n").alias("amt_sd"), F.col("mad_n").alias("amt_mad"),
                  F.col("spread_n").alias("spread"), F.col("p10a_n").alias("amt_p10"), F.col("p90a_n").alias("amt_p90"),
                  "s12", (F.col("s12") / F.col("_t12")).alias("s12_share"),
                  F.size("mo2").alias("active_months"), F.col("c_amt").alias("amt"), F.col("c_ntxn").alias("ntxn"),
                  "n_exp", "exp_amt", "exp_lo", "exp_hi", F.col("meth_p").alias("exp_method"), F.col("bias_p").alias("exp_bias"),
                  F.col("lvl_p").alias("cal_level"), "exp_med3", "exp_mean", "exp_med", "exp_last", "shortfall",
                  "became_regular", "late", "early", "skipped", "days_late", "overdue", "stopped", "payer_wide", "resumed",
                  "amt_low", "amt_high", "next_dt", "next_lo_dt", "next_hi_dt", F.col("fn1").alias("fc1_n"), "fc1_amt", "fc1_lo",
                  "fc1_hi", F.col("meth_n").alias("fc_method"), F.col("bias_n").alias("fc_bias"), "p_continue", "fc3_amt"))
    S.write.mode("overwrite").parquet(f"{LOC['stream']}/month={m}")
    # 4c month table: dollars by the class each stream had at the start of the month (cadence if regular, else irregular);
    #     live streams and forecasts by the cadence at the end of the month
    cnt = lambda c: F.sum(F.col(c).cast("int")).cast("long")
    cls = F.when(F.col("reg_p"), F.col("cad_p")).otherwise(F.lit("irregular"))
    a1 = (J.withColumn("cadence", cls).groupBy("cust", "dir", "side", "cadence")
           .agg(F.sum("c_amt").alias("amt"), F.sum("c_ntxn").alias("ntxn"), F.sum(F.col("act").cast("int")).cast("long").alias("n_active"),
                cnt("due").alias("n_due"), F.sum(F.when(F.col("due"), F.col("c_amt")).otherwise(0.0)).alias("amt_due"),
                F.sum("exp_amt").alias("exp_amt"),
                F.sum(F.when(F.col("overdue") | F.col("stopped"), F.col("exp_amt"))).alias("at_risk_amt"),
                F.sum("shortfall").alias("shortfall"),
                *[cnt(c).alias(f"n_{c}") for c in ("late", "early", "skipped", "overdue", "stopped", "payer_wide", "resumed",
                                                   "amt_low", "amt_high")]))
    a2 = (J.where(F.col("live") | F.col("became_regular")).withColumn("cadence", F.col("cad_n")).groupBy("cust", "dir", "side", "cadence")
           .agg(cnt("live").alias("n_live"), cnt("became_regular").alias("n_became_regular"),
                F.sum(F.when(F.col("live"), F.col("fc1_amt") * F.coalesce(F.col("p_continue"), F.lit(1.0)))).alias("fc1_amt"),
                F.sum("fc3_amt").alias("fc3_amt"),
                F.sum(F.when(F.col("live"), F.col("s12"))).alias("s12_live")))
    MT = a1.join(a2, ["cust", "dir", "side", "cadence"], "full")
    MT.write.mode("overwrite").parquet(f"{LOC['month']}/month={m}")
    # 4d total table: one row per customer × direction × side, with customer-level intervals
    T = (J.groupBy("cust", "dir", "side")
          .agg(F.sum("c_amt").alias("amt_all"), F.sum("c_ntxn").alias("ntxn_all"), F.sum(F.col("act").cast("int")).cast("long").alias("n_active"),
               F.sum(F.when(F.col("due"), F.col("c_amt")).otherwise(0.0)).alias("amt_due"), cnt("due").alias("n_due"),
               cnt("live").alias("n_live"), F.sum("exp_amt").alias("exp_amt"),
               F.sum(F.when(F.col("overdue") | F.col("stopped"), F.col("exp_amt"))).alias("at_risk_amt"), F.sum("shortfall").alias("shortfall"),
               *[cnt(c).alias(f"n_{c}") for c in ("late", "early", "skipped", "overdue", "stopped", "payer_wide", "resumed",
                                                  "amt_low", "amt_high", "became_regular")],
               F.sum(F.when(F.col("live"), F.col("fc1_amt") * F.coalesce(F.col("p_continue"), F.lit(1.0)))).alias("fc1_amt"),
               F.sum("fc3_amt").alias("fc3_amt"),
               F.sum(F.when(F.col("live"), F.col("s12"))).alias("s12_live"), F.max(F.when(F.col("live"), F.col("s12"))).alias("_top"))
          .withColumn("coverage", F.col("amt_due") / F.col("amt_all"))
          .withColumn("received_share", F.col("amt_due") / F.col("exp_amt"))
          .withColumn("top_share", F.col("_top") / F.col("s12_live")).drop("_top"))
    def cust_q(cal, nb_col, a, lo_c, hi_c):
        """Customer-level interval: a × percentiles of the matching direction × side × number-of-streams group."""
        nonlocal T
        cc = None if cal is None else cal[cal.n >= CALIB_MIN_N][["dir", "side", "nb", "q_lo", "q_hi"]]
        if cc is None or not len(cc):
            T = T.withColumn(lo_c, F.lit(None).cast("double")).withColumn(hi_c, F.lit(None).cast("double"))
            return
        cc = spark.createDataFrame([tuple(_native(v) for v in r) for r in cc.itertuples(index=False)],
                                   "dir string, side string, nb string, q_lo double, q_hi double")
        T = (T.withColumn("nb", nbucket(F.col(nb_col))).join(F.broadcast(cc), ["dir", "side", "nb"], "left")
              .withColumn(lo_c, F.col(a) * F.col("q_lo")).withColumn(hi_c, F.col(a) * F.col("q_hi")).drop("nb", "q_lo", "q_hi"))
    cust_q(ccal, "n_due", "exp_amt", "exp_lo", "exp_hi")
    cust_q(fcal, "n_live", "fc1_amt", "fc1_lo", "fc1_hi")
    T = T.select("cust", "dir", "side", "amt_all", "ntxn_all", "n_active", "amt_due", "coverage", "n_due", "exp_amt", "exp_lo", "exp_hi",
                 "received_share", "at_risk_amt", "shortfall", "n_live", "fc1_amt", "fc1_lo", "fc1_hi", "fc3_amt", "s12_live", "top_share",
                 *[f"n_{c}" for c in ("became_regular", "late", "early", "skipped", "overdue", "stopped", "payer_wide", "resumed",
                                      "amt_low", "amt_high")])
    T.write.mode("overwrite").parquet(f"{LOC['total']}/month={m}")
    # 4e calibration rows used for this month
    CAL_COLS = (["level", "tier"] + CAL_KEYS + ["bucket", "n", "method", "bias"] + [f"err_{k}" for k in METHODS]
                + ["q_lo", "q_mid", "q_hi", "stop_rate"])
    cal_rows = []
    def add_rows(df, level, **fixed):
        if df is None:
            return
        for _, r in df.iterrows():
            row = {c: None for c in CAL_COLS}
            row.update({k: "*" for k in CAL_KEYS})
            row.update({k: v for k, v in r.items() if k in row}); row.update(fixed); row["level"] = level
            for k in CAL_KEYS:
                v = _native(row[k])
                row[k] = "*" if v is None else str(v).lower() if isinstance(v, bool) else str(v)
            cal_rows.append(tuple(_native(row[c]) for c in CAL_COLS))
    add_rows(scal[scal.n >= CALIB_MIN_N] if scal is not None else None, "stream")
    add_rows(hz.rename(columns={"hazard": "stop_rate"}) if hz is not None else None, "stop rate")
    add_rows(ccal.rename(columns={"nb": "bucket"}) if ccal is not None else None, "customer: this month")
    add_rows(fcal.rename(columns={"nb": "bucket"}) if fcal is not None else None, "customer: next month")
    (spark.createDataFrame(cal_rows, "level string, tier int, " + ", ".join(f"`{k}` string" for k in CAL_KEYS)
                           + ", bucket string, n long, method string, bias double, " + ", ".join(f"err_{k} double" for k in METHODS)
                           + ", q_lo double, q_mid double, q_hi double, stop_rate double")
          .withColumn("prior_months", F.lit(n_prior)).coalesce(1).write.mode("overwrite").parquet(f"{LOC['calib']}/month={m}"))
    steps["outputs"] = round((time.time() - t0) / 60 - sum(steps.values()), 2)
    # 5 reconcile: every dollar of the month is in the total table (pair table, int + ext)
    got = spark.read.parquet(f"{LOC['total']}/month={m}").agg(F.sum("amt_all")).first()[0] or 0.0
    want = float(spark.table(T_PAIR).where((F.col("month") == m) & F.col("scope").isin("int", "ext")).agg(F.sum("amount")).first()[0] or 0)
    if abs(got - want) > max(1.0, 1e-9 * abs(want)):
        raise AssertionError(f"{m}: dollars do not reconcile — total table {got:,.2f} vs pair table {want:,.2f}")
    n_s = spark.read.parquet(f"{LOC['stream']}/month={m}").count()
    write_json(f"{LOC['total']}/month={m}/_PKG_BUILD", {"sig": SIG, "fp": month_fp(m), "build_id": build_id, "prev_build": prev_build,
                                                        "version": PER_VERSION, "built_at": dt.datetime.now().isoformat(timespec="seconds")})
    delete(work)
    steps["total"] = round((time.time() - t0) / 60, 2)
    return {"month": m, "stream_rows": n_s, "dollars": got, "calib_months": n_prior, **steps}

LOG = []
prev_build, building, LK = None, False, None
for m in MONTHS:
    mk = marker(m)
    if not building and month_why(m, mk, prev_build) is None:
        prev_build = mk["build_id"]                   # done: keep it
        continue
    if not building:                                  # first month to build in this run
        check_prev_state(m)
        building = True
    if LK is None:                                    # counterparty id → entity, once per run
        lk_table().write.mode("overwrite").parquet(f"{WORK}/lk")
        LK = spark.read.parquet(f"{WORK}/lk")
    bid = hashlib.md5(f"{m}{time.time()}".encode()).hexdigest()[:12]
    try:
        LOG.append(build_month(m, prev_build, bid))
    except Exception as e:
        LOG.append({"month": m, "error": str(e).splitlines()[0][:200]})
        out(sec(f"Stopped at {m}. Finished months are kept: run §2 again (or the whole notebook) and it resumes at {m}"
                + (" — set REBUILD_FROM = None first, or the months already rebuilt are rebuilt again" if REBUILD_FROM else ""),
                tbl(pd.DataFrame(LOG))))
        raise
    prev_build = bid
    # keep the KEEP_STATES states up to m; states of later months belong to an older build (every month after m is
    # rebuilt now that m has a new build id), so they go too
    keep_ = set(month_range(month_add(m, -(KEEP_STATES - 1)), m))
    for sm in [x.split("=", 1)[1] for x in ls_dirs(STATE) if x.startswith("month=")]:
        if sm not in keep_:
            delete(f"{STATE}/month={sm}")

# table definitions (external; re-created when the columns change) and partitions
def register(k):
    parts_ = sorted(x for x in ls_dirs(LOC[k]) if x.startswith("month="))
    if not parts_:
        return
    sch = spark.read.parquet(f"{LOC[k]}/{parts_[-1]}").schema
    cols = ", ".join(f"`{f.name}` {f.dataType.simpleString()}" for f in sch.fields)
    if table_exists(TBL[k]) and [c for c in spark.table(TBL[k]).columns if c != "month"] != [f.name for f in sch.fields]:
        spark.sql(f"DROP TABLE {TBL[k]}")
    spark.sql(f"CREATE EXTERNAL TABLE IF NOT EXISTS {TBL[k]} ({cols}) PARTITIONED BY (month string) STORED AS PARQUET LOCATION '{LOC[k]}'")
    spark.sql(f"ALTER TABLE {TBL[k]} RECOVER PARTITIONS")
    spark.sql(f"REFRESH TABLE {TBL[k]}")
for k in ("stream", "month", "total", "calib"):
    register(k)
delete(f"{WORK}/lk")
_S2_OK = True
lg = pd.DataFrame(LOG)
out(sec(f"§2 Built {len(LOG)} month(s)" if LOG else "§2 Nothing to build: every month is done for this signature",
        tbl(lg, dec=tuple(c for c in lg.columns if c in ("calib", "streams", "outputs", "total", "dollars")))) if LOG else "",
    para("Table definitions refreshed: run INVALIDATE METADATA in Impala for " + ", ".join(TBL[k] for k in ("stream", "month", "total", "calib")) + "."))

## §3 QA

On the latest month (and the `QA_MONTHS` before it for accuracy):
- **a** coverage: share of dollars that were expected (from streams due), by direction × side, TM and non-TM;
- **b** live regular streams by side × cadence and their last-12-month dollars;
- **c** events this month by direction × cadence;
- **d** expectation compared on past months (stream level, stopped streams left out): the expectation in use (chosen
  predictor × bias correction) against projected dates × each raw predictor; which predictor the groups chose, their
  bias correction and how fine a group the streams got;
- **e** interval coverage (stream level by cadence × edge, and customer level; should be about 90%);
- **f** forecasts against what came: next month (customer level, stop rates applied) and the next `FORECAST_MONTHS`
  months (stream level);
- **g** payers with the most payer-wide stops this month.

In [ ]:
# ---- §3 QA ---------------------------------------------------------------------------------------
assert globals().get("_S2_OK"), "§2 did not finish: fix the problem it shows, then run all cells again"

QM = MONTHS[-1]
qa_months = [x for x in month_range(month_add(QM, -(QA_MONTHS - 1)), QM) if x >= MONTHS[0]]
TS = spark.table(TBL["stream"]); TT = spark.table(TBL["total"]); TM_ = spark.table(TBL["month"])
tm = spark.table(T_CUST).select(F.col("mdm_id").alias("cust"), F.coalesce(F.col("is_tm").cast("boolean"), F.lit(False)).alias("tm"))
SIDE_ORDER = {"ext": 0, "int": 1, "own": 2}
# a coverage
cov = (TT.where(F.col("month") == QM).join(tm, "cust", "left")
         .groupBy("dir", "side", F.coalesce("tm", F.lit(False)).alias("tm"))
         .agg(F.countDistinct("cust").alias("customers"), (F.sum("amt_all") / 1e6).alias("dollars (USD mn)"),
              (F.sum("amt_due") / F.sum("amt_all")).alias("expected share of dollars"),
              F.avg((F.col("n_due") > 0).cast("double")).alias("customers with a due stream"),
              F.percentile_approx("coverage", 0.5, 1000).alias("median customer coverage"),
              F.percentile_approx("coverage", 0.9, 1000).alias("p90 customer coverage"))
         .toPandas().sort_values(["dir", "side", "tm"]))
# b live streams
lv = (TS.where((F.col("month") == QM) & F.col("live")).groupBy("side", "cadence")
        .agg(F.count("*").alias("live streams"), (F.sum("s12") / 1e6).alias("last 12 months (USD mn)"),
             F.percentile_approx("spread", 0.5, 1000).alias("median spread"),
             F.avg((F.col("spread") == 0).cast("double")).alias("fixed amount"))
        .toPandas())
lv["_o"] = lv.side.map(SIDE_ORDER) * 10 + lv.cadence.map({c: i for i, c in enumerate(CAD_GROUPS)})
lv = lv.sort_values("_o").drop(columns="_o")
# c events
EV = ["n_due", "n_late", "n_early", "n_skipped", "n_overdue", "n_stopped", "n_payer_wide", "n_resumed", "n_amt_low", "n_amt_high", "n_became_regular"]
ev = (TM_.where((F.col("month") == QM) & (F.col("cadence") != "irregular")).groupBy("dir", "cadence")
         .agg(*[F.sum(c).cast("long").alias(c[2:]) for c in EV]).toPandas())
ev["_o"] = ev.dir.map({"in": 0, "out": 1}) * 10 + ev.cadence.map({c: i for i, c in enumerate(CAD_GROUPS)})
ev = ev.sort_values("_o").drop(columns="_o")
# d methods
METH = {"in use": "exp_amt", "median of last 3": "exp_med3", "mean": "exp_mean", "median": "exp_med", "last": "exp_last"}
hist = TS.where(F.col("month").isin(qa_months) & F.col("due") & ~F.col("stopped"))
rows = []
for nm_, c in METH.items():
    e_ = F.col(c)
    r = (hist.where(e_.isNotNull()).groupBy("cadence_p")
             .agg((F.sum("amt") / F.sum(e_)).alias("received / expected"),
                  (F.sum(F.abs(F.col("amt") - e_)) / F.sum("amt")).alias("dollar error"),
                  F.avg(F.when(e_ > 0, (F.abs(F.col("amt") - e_) <= 0.1 * e_).cast("double")).otherwise((F.col("amt") == 0).cast("double"))).alias("within ±10%"))
             .toPandas())
    r["method"] = nm_; rows.append(r)
md_ = pd.concat(rows)
md_ = md_.pivot(index="cadence_p", columns="method", values=["dollar error", "within ±10%", "received / expected"])
md_.columns = [f"{b}: {a}" for a, b in md_.columns]
md_ = md_.reindex([c for c in CAD_GROUPS if c in md_.index]).reset_index().rename(columns={"cadence_p": "cadence"})
md_ = md_[["cadence"] + [f"{n}: {mt}" for mt in ["dollar error", "within ±10%", "received / expected"] for n in METH]]
# which predictor the streams got, their bias correction and calibration level (due streams, latest month)
cur_ = TS.where((F.col("month") == QM) & F.col("due"))
mu = (cur_.groupBy("cadence_p").agg(F.count("*").alias("due streams"),
                                    *[F.avg((F.col("exp_method") == k).cast("double")).alias(f"uses {k}") for k in METHODS],
                                    F.percentile_approx("exp_bias", [0.1, 0.5, 0.9], 1000).alias("_b"),
                                    (F.sum(F.col("exp_amt")) / F.sum(F.col("exp_amt") / F.col("exp_bias"))).alias("bias, dollar-weighted"),
                                    *[F.avg(F.coalesce(F.col("cal_level") == t, F.lit(False)).cast("double")).alias(f"level {t}")
                                      for t in range(1, len(CALIB_LEVELS) + 1)],
                                    F.avg(F.col("cal_level").isNull().cast("double")).alias("no group"))
          .withColumn("bias p10", F.col("_b")[0]).withColumn("bias p50", F.col("_b")[1]).withColumn("bias p90", F.col("_b")[2]).drop("_b")
          .toPandas().rename(columns={"cadence_p": "cadence"}))
mu = mu.assign(_o=mu.cadence.map({c: i for i, c in enumerate(CAD_GROUPS)})).sort_values("_o").drop(columns="_o")
mu = mu[["cadence", "due streams"] + [f"uses {k}" for k in METHODS] + ["bias p10", "bias p50", "bias p90", "bias, dollar-weighted"]
        + [f"level {t}" for t in range(1, len(CALIB_LEVELS) + 1)] + ["no group"]]
# e interval coverage
ins = lambda a, lo, hi: ((F.col(a) >= F.col(lo)) & (F.col(a) <= F.col(hi))).cast("double")
ic_s = (hist.where(F.col("exp_lo").isNotNull()).groupBy("cadence_p", "edge_p")
            .agg(F.count("*").alias("stream-months"), F.avg(ins("amt", "exp_lo", "exp_hi")).alias("inside"),
                 F.avg((F.col("amt") < F.col("exp_lo")).cast("double")).alias("below"),
                 F.avg((F.col("amt") > F.col("exp_hi")).cast("double")).alias("above"),
                 F.percentile_approx((F.col("exp_hi") - F.col("exp_lo")) / F.col("exp_amt"), 0.5, 1000).alias("width"))
            .toPandas().rename(columns={"cadence_p": "cadence", "edge_p": "edge"}))
ic_s = ic_s.assign(_o=ic_s.cadence.map({c: i for i, c in enumerate(CAD_GROUPS)}) * 2 + ic_s.edge.astype(int)).sort_values("_o").drop(columns="_o")
ic_c = (TT.where(F.col("month").isin(qa_months) & F.col("exp_lo").isNotNull() & (F.col("exp_amt") > 0)).groupBy("dir", "side")
          .agg(F.count("*").alias("customer-months"), F.avg(ins("amt_due", "exp_lo", "exp_hi")).alias("inside"),
               F.avg((F.col("amt_due") < F.col("exp_lo")).cast("double")).alias("below"),
               F.avg((F.col("amt_due") > F.col("exp_hi")).cast("double")).alias("above"),
               F.percentile_approx((F.col("exp_hi") - F.col("exp_lo")) / F.col("exp_amt"), 0.5, 1000).alias("width"))
          .toPandas().sort_values(["dir", "side"]))
# f forecasts: next month (customer level: fc1 at m−1 against the dollars of streams due at m) and next quarter
#   (stream level: fc3 at m−3 against the stream's dollars in the 3 months after)
prs = [(month_add(x, -1), x) for x in qa_months if month_add(x, -1) >= MONTHS[0]]
f1 = None
for a_, b_ in prs:
    j = (TT.where(F.col("month") == a_).select("cust", "dir", "side", "fc1_amt", "fc1_lo", "fc1_hi")
           .join(TT.where(F.col("month") == b_).select("cust", "dir", "side", "amt_due"), ["cust", "dir", "side"], "left")
           .fillna({"amt_due": 0.0}).where(F.col("fc1_amt") > 0))
    f1 = j if f1 is None else f1.unionByName(j)
fc1 = (f1.groupBy("dir", "side").agg(F.count("*").alias("customer-months"), (F.sum("amt_due") / F.sum("fc1_amt")).alias("came / forecast"),
                                     (F.sum(F.abs(F.col("amt_due") - F.col("fc1_amt"))) / F.sum("amt_due")).alias("dollar error"),
                                     F.avg((F.abs(F.col("amt_due") - F.col("fc1_amt")) <= 0.1 * F.col("fc1_amt")).cast("double")).alias("within ±10%"),
                                     F.avg(F.when(F.col("fc1_lo").isNotNull(), ins("amt_due", "fc1_lo", "fc1_hi"))).alias("inside interval"))
         .toPandas().sort_values(["dir", "side"])) if f1 is not None else pd.DataFrame()
q3 = [x for x in qa_months if month_add(x, -3) >= MONTHS[0] and x <= MONTHS[-1]]
f3 = None
for b_ in q3:
    a_ = month_add(b_, -3)
    nxt = TS.where(F.col("month").isin(month_range(month_add(a_, 1), b_))).groupBy(*K).agg(F.sum("amt").alias("came"))
    j = (TS.where((F.col("month") == a_) & F.col("live")).select(*K, "side", "cadence", "fc3_amt")
           .join(nxt, K, "left").fillna({"came": 0.0}))
    f3 = j if f3 is None else f3.unionByName(j)
fc3 = (f3.groupBy("side", "cadence").agg(F.count("*").alias("streams"), (F.sum("came") / F.sum("fc3_amt")).alias("came / forecast"),
                                          (F.sum(F.abs(F.col("came") - F.col("fc3_amt"))) / F.sum("came")).alias("dollar error"),
                                          F.avg((F.abs(F.col("came") - F.col("fc3_amt")) <= 0.1 * F.col("fc3_amt")).cast("double")).alias("within ±10%"))
         .toPandas()) if f3 is not None else pd.DataFrame()
if len(fc3):
    fc3["_o"] = fc3.side.map(SIDE_ORDER) * 10 + fc3.cadence.map({c: i for i, c in enumerate(CAD_GROUPS)})
    fc3 = fc3.sort_values("_o").drop(columns="_o")
# g payer-wide stops
pwt = (TS.where((F.col("month") == QM) & F.col("payer_wide")).groupBy("other").agg(F.count("*").alias("stopped customers"),
                                                                                     (F.sum("exp_amt") / 1e3).alias("expected (USD k)"))
         .orderBy(F.desc("stopped customers")).limit(15).toPandas())
PCT3 = ("expected share of dollars", "customers with a due stream", "median customer coverage", "p90 customer coverage", "median spread",
        "fixed amount", "inside", "below", "above", "width", "dollar error", "within ±10%", "inside interval",
        *[f"uses {k}" for k in METHODS], *[f"level {t}" for t in range(1, len(CALIB_LEVELS) + 1)], "no group")
out(sec(f"§3a Coverage {QM}: share of dollars that came from streams due this month", tbl(cov, pct=PCT3, dec=("dollars (USD mn)",))),
    sec(f"§3b Live regular streams at the end of {QM}", tbl(lv, pct=PCT3, dec=("last 12 months (USD mn)",))),
    sec(f"§3c Events in {QM} (streams)", tbl(ev)),
    sec(f"§3d Expectation in use vs projected dates × each raw predictor, {qa_months[0]} → {qa_months[-1]} (due streams, stopped left out)",
        tbl(md_, pct=tuple(c for c in md_.columns if "error" in c or "within" in c), dec=tuple(c for c in md_.columns if "received" in c))),
    sec(f"§3d Predictor, bias correction and calibration level of the due streams in {QM}",
        tbl(mu, pct=PCT3, dec=("bias p10", "bias p50", "bias p90", "bias, dollar-weighted"))
        + para("Level 1 = the finest group (" + " × ".join(CALIB_LEVELS[0]) + "), up to level "
             + f"{len(CALIB_LEVELS)} (" + " × ".join(CALIB_LEVELS[-1]) + "); no group = default predictor, no correction.")),
    sec("§3e Interval coverage, stream level by cadence × edge (90% nominal)", tbl(ic_s, pct=PCT3)),
    sec("§3e Interval coverage, customer level (90% nominal)", tbl(ic_c, pct=PCT3)),
    sec("§3f Next-month forecast against what came (customer level, stop rates applied)", tbl(fc1, pct=PCT3, dec=("came / forecast",)) if len(fc1) else para("—")),
    sec(f"§3f Next-{FORECAST_MONTHS}-month forecast against what came (stream level)", tbl(fc3, pct=PCT3, dec=("came / forecast",)) if len(fc3) else para("—")),
    sec(f"§3g Payers with the most payer-wide stops in {QM}", tbl(pwt, dec=("expected (USD k)",)) if len(pwt) else para("none")))

## §4 Dictionary and housekeeping

In [ ]:
# ---- §4 Dictionary and housekeeping ---------------------------------------------------------------
assert globals().get("_S2_OK"), "§2 did not finish: fix the problem it shows, then run all cells again"

DICT = []
def col(t, c, d):
    DICT.append({"table": TBL[t], "column": c, "definition": d})
for t in ("stream", "month", "total"):
    col(t, "cust", "customer mdm_id"); col(t, "dir", "in = received, out = paid")
    col(t, "side", "ext = external counterparty; int = another PNC customer (book to book); own = the customer's own-name account at another bank (self-pay match)")
    col(t, "month", "YYYY-MM")
for c, d in [("scope", "int / ext as in the customer-metrics pair table (own is ext with the self-pay match)"),
             ("other", "counterparty: entity (ent_l2 of the lowest map row) for ext / own, mdm_id for int"),
             ("raw_id", "counterparty id of the largest payment on the stream's last payment day"),
             ("bank", "main canonical bank of the counterparty (pair table, latest month the stream was paid)"),
             ("o_class", "name class of the counterparty (pair table)"), ("rail", "rail of the largest payment on the last payment day"),
             ("cadence", "cadence at the end of the month (live streams), else at the start"), ("cadence_p", "cadence at the start of the month"),
             ("rail_p", "rail at the start of the month (calibration key; ? = unknown)"),
             ("amt_band_p", f"median payment of the history at the start of the month, in bands {AMT_BANDS} dollars (calibration key)"),
             ("spread_bin_p", "spread of the history at the start of the month, binned (calibration key)"),
             ("edge_p", f"a projected payment date within {EDGE_DAYS} days of the start or end of the month, where a shift of a few days moves it into the month next to it (calibration key)"),
             ("status", "end of month: live, overdue (late, not yet stopped), stopped (this month), resumed, not regular"),
             ("due", "regular and not stopped at the end of the month before: a payment was expected"),
             ("live", "regular and not stopped at the end of the month"),
             ("n_pay", f"payment days in the last {REG_WINDOW} months (history kept: latest {REG_MAX_DAYS})"),
             ("first_dt", "first payment day in the history kept"), ("last_dt", "last payment day"),
             ("med_gap", "median gap between payment days (calendar days)"), ("gap_p10", "p10 of the gaps (daily: business days)"),
             ("gap_p90", "p90 of the gaps (daily: business days)"), ("ontime_share", "share of gaps on time"),
             ("pay_dom", "usual day of the month (median of the last 3 payments; anchor of the projected dates)"),
             ("daily_fill", "daily streams: share of business days with a payment over the last 3 months"),
             ("amt_med", f"median of the last {REG_MAX_DAYS} payment days"), ("amt_med3", f"median of the last {AMT_RECENT} payment days"),
             ("amt_last", "last payment day's dollars"), ("amt_mean", "mean of the history"), ("amt_sd", "standard deviation of the history"),
             ("amt_mad", "median absolute difference from amt_med"), ("spread", "amt_mad / amt_med: 0 = fixed amount; the larger, the less the amount can be trusted"),
             ("amt_p10", "p10 of the history"), ("amt_p90", "p90 of the history"),
             ("s12", "the stream's dollars in the last 12 months"), ("s12_share", "s12 / all the customer's dollars in that direction in the last 12 months (dependence)"),
             ("active_months", f"months with a payment in the last {REG_WINDOW}"),
             ("amt", "dollars this month"), ("ntxn", "transactions this month"),
             ("n_exp", "payments expected this month (projected dates; daily: business days × daily_fill), due streams"),
             ("exp_amt", "expected dollars this month, if the stream continues = n_exp × the predictor in exp_method × exp_bias"),
             ("exp_lo", f"exp_amt × p{INTERVAL[0] * 100:g} of past received ÷ corrected expected in the stream's calibration group"),
             ("exp_hi", f"exp_amt × p{INTERVAL[1] * 100:g} of the same"),
             ("exp_method", f"predictor of the amount per payment: {', '.join(METHODS)} (median of the last {AMT_RECENT}, mean, median of the history, last payment); the one with the lowest past dollar error in the stream's calibration group"),
             ("exp_bias", f"bias correction: past received ÷ expected with that predictor in the group, kept within {list(BIAS_RANGE)}; 1 = no group"),
             ("cal_level", f"calibration group used: 1 = finest ({' × '.join(CALIB_LEVELS[0])}) … {len(CALIB_LEVELS)} = coarsest ({' × '.join(CALIB_LEVELS[-1])}); NULL = none (default predictor, no correction, no interval)"),
             ("exp_med3", f"n_exp × median of the last {AMT_RECENT} (raw, no correction)"), ("exp_mean", "n_exp × mean (raw)"),
             ("exp_med", f"n_exp × median of the last {REG_MAX_DAYS} (raw)"), ("exp_last", "n_exp × last payment (raw)"),
             ("shortfall", "max(exp_amt − amt, 0), due streams"),
             ("became_regular", "regular at the end of the month, not at the start"),
             ("late", "a payment came after the usual gap + tolerance (< SKIP_MULT × gap), non-daily"),
             ("early", "a payment came before the usual gap − tolerance, non-daily"),
             ("skipped", f"a payment came after ≥ {SKIP_MULT} × the usual gap (daily: ≥ 2 business days)"),
             ("days_late", "largest delay this month beyond the usual gap (days; daily: business days beyond the next)"),
             ("overdue", "due, no payment yet beyond the usual gap + tolerance at month end, not stopped"),
             ("stopped", f"due and silent for {SKIP_MULT} × the usual gap at month end (daily: {DAILY_STOP_BD} business days)"),
             ("payer_wide", f"stopped, and the payer stopped ≥ {MASS_STOP_MIN} customers this month (incoming)"),
             ("resumed", "was stopped, paid again this month"),
             ("amt_low", f"a payment below min(2 × p10 − median, median × {1 - FIXED_PCT}), due non-daily streams"),
             ("amt_high", f"a payment above max(2 × p90 − median, median × {1 + FIXED_PCT}), due non-daily streams"),
             ("next_dt", "next projected payment date (live streams)"),
             ("next_lo_dt", "last_dt + gap_p10 (non-daily)"), ("next_hi_dt", "last_dt + gap_p90 (non-daily)"),
             ("fc1_n", "payments expected next month"),
             ("fc1_amt", "dollars expected next month if the stream continues (predictor and bias of the group at the end of the month)"),
             ("fc1_lo", f"fc1_amt × p{INTERVAL[0] * 100:g} of past received ÷ corrected expected (group at the end of the month)"),
             ("fc1_hi", f"fc1_amt × p{INTERVAL[1] * 100:g} of the same"),
             ("fc_method", "predictor used for fc1_amt and fc3_amt"), ("fc_bias", "bias correction used for fc1_amt and fc3_amt"),
             ("p_continue", "1 − past monthly stop rate (side × cadence × rail)"),
             ("fc3_amt", f"expected dollars over the next {FORECAST_MONTHS} months, each month × p_continue^k")]:
    col("stream", c, d)
for c, d in [("cadence", "class at the start of the month for dollars, events and expectation (cadence if regular, else irregular); "
                         "at the end of the month for n_live, n_became_regular, fc1_amt, fc3_amt, s12_live"),
             ("amt", "dollars this month"), ("ntxn", "transactions"), ("n_active", "streams with a payment this month"),
             ("n_due", "due streams"), ("amt_due", "dollars of due streams"), ("exp_amt", "expected dollars (due streams)"),
             ("at_risk_amt", "expected dollars of streams overdue or stopped at month end"), ("shortfall", "sum of stream shortfalls"),
             ("n_late … n_amt_high", "streams with the event this month"), ("n_live", "live streams at month end"),
             ("n_became_regular", "streams that became regular"), ("fc1_amt", "next month: stream fc1_amt × p_continue (stop rates applied)"),
             ("fc3_amt", f"next {FORECAST_MONTHS} months, stop rates applied"), ("s12_live", "last-12-month dollars of live streams")]:
    col("month", c, d)
for c, d in [("amt_all", "all dollars this month (every stream, regular or not)"), ("ntxn_all", "all transactions"),
             ("n_active", "streams with a payment this month"), ("amt_due", "dollars of streams due this month"),
             ("coverage", "amt_due / amt_all: the share of the month's dollars that was expected"),
             ("n_due", "due streams"), ("exp_amt", "expected dollars"),
             ("exp_lo", "exp_amt × p5 of past customer-level amt_due ÷ exp_amt (direction × side × number of due streams)"),
             ("exp_hi", "exp_amt × p95 of the same"), ("received_share", "amt_due / exp_amt"),
             ("at_risk_amt", "expected dollars of streams overdue or stopped"), ("shortfall", "sum of stream shortfalls"),
             ("n_live", "live streams at month end"), ("fc1_amt", "next month from live streams: stream fc1_amt × p_continue (stop rates applied)"),
             ("fc1_lo", f"fc1_amt × p{INTERVAL[0] * 100:g} of past customer-level what came ÷ fc1_amt of the month before (direction × side × number of live streams)"),
             ("fc1_hi", f"fc1_amt × p{INTERVAL[1] * 100:g} of the same"),
             ("fc3_amt", f"next {FORECAST_MONTHS} months, stop rates applied"), ("s12_live", "last-12-month dollars of live streams"),
             ("top_share", "largest live stream's s12 / s12_live"), ("n_<event>", "streams with the event this month")]:
    col("total", c, d)
for c, d in [("level", "stream (predictor, bias, interval percentiles per group), stop rate, customer: this month (interval of exp_amt), customer: next month (interval of fc1_amt)"),
             ("tier", f"stream groups: 1 = finest ({' × '.join(CALIB_LEVELS[0])}) … {len(CALIB_LEVELS)} = coarsest; a stream uses the finest group that has one"),
             ("dir / side / cadence / rail / amt_band / spread_bin / edge", "group keys at the start of the month (* = not a key of this level)"),
             ("bucket", "customer levels: number of due (this month) or live (next month) streams"),
             ("n", "rows behind the numbers (sample of CALIB_PCT % of customers)"),
             ("method", "predictor with the lowest dollar error after bias correction"), ("bias", "received ÷ expected with that predictor"),
             ("err_<predictor>", "dollar error Σ|received − bias × expected| ÷ Σ received of each predictor after its own bias correction"),
             ("q_lo / q_mid / q_hi", f"p{INTERVAL[0] * 100:g} / p50 / p{INTERVAL[1] * 100:g} of received ÷ corrected expected (customer: received ÷ expected or ÷ forecast)"),
             ("stop_rate", "share of due streams that stopped in a month"), ("prior_months", "published months the calibration used"),
             ("month", "month the calibration was used for (from earlier months only)")]:
    col("calib", c, d)
dd_ = pd.DataFrame(DICT)
(spark.createDataFrame(dd_).coalesce(1).write.mode("overwrite").parquet(LOC["dictionary"]))
if not table_exists(TBL["dictionary"]):
    spark.sql(f"CREATE EXTERNAL TABLE {TBL['dictionary']} (`table` string, `column` string, definition string) STORED AS PARQUET LOCATION '{LOC['dictionary']}'")
spark.sql(f"REFRESH TABLE {TBL['dictionary']}")
sizes = pd.DataFrame([{"table / folder": TBL.get(k, k), "months": len([x for x in ls_dirs(LOC[k]) if x.startswith("month=")]),
                       "GB": size_gb(LOC[k])} for k in ("stream", "month", "total", "calib")]
                     + [{"table / folder": "stream state (working, latest months only)",
                         "months": len([x for x in ls_dirs(STATE) if x.startswith("month=")]), "GB": size_gb(STATE)}])
sizes["GB per month"] = sizes.GB / sizes.months.clip(lower=1)
out(sec(f"§4 Dictionary: {len(dd_)} entries in {TBL['dictionary']}", ""),
    sec("Storage (logical; HDFS keeps 3 copies)", tbl(sizes, dec=("GB", "GB per month"))))